# Sitio web del Trabajo 1 – Balance hídrico de largo plazo del río Cauca hasta Hidroituango

Este notebook **arma una página web con todos los resultados** que generaron los notebooks de los puntos 2 a 14:
figuras, tablas y cifras, organizados en las 14 secciones del enunciado.
- Los mapas se amplían con un clic.
- Los gráficos son interactivos: al pasar el cursor (o tocar en el celular) muestran los valores, y cada uno tiene
  «Ver los datos del gráfico». No usan librerías externas, así que funcionan sin internet.

**Antes de correrlo:** corra los notebooks de los puntos 3 a 14 en esta misma carpeta (`04_PY`). Si falta algún
resultado, la página lo indica en la sección correspondiente y el resto se arma igual.

**Salida:** la carpeta `sitio/`, con `index.html` y las imágenes. Para verla, abra `sitio/index.html` con doble clic.
Al final de este notebook están los pasos para publicarla en GitHub Pages.

## 1. Parámetros

In [13]:
import os, re, json, shutil, html, datetime
import numpy as np
import pandas as pd

# ----------------------------- PARÁMETROS -----------------------------
RESULTADOS = "."                       # carpeta donde están las salidas de los notebooks (04_PY)
SALIDA = "sitio"                       # carpeta de la página web
TITULO = "Balance hídrico de largo plazo · Río Cauca hasta Hidroituango"
SUBTITULO = "Trabajo 1 de Hidroclimatología · Universidad Nacional de Colombia · 1981–2010"
AUTORES = "Daniela Isabel Sierra Herrera"
CIERRE = (7.1306, -75.6635)           # sitio de presa de Hidroituango (lat, lon)
MAPA_DELIMITACION = ""                 # opcional: imagen del mapa del punto 2 exportada de QGIS (p. ej. "mapa_cuenca_qgis.png")
ANCHO_MAX_IMG = 1800                   # las imágenes se reducen a este ancho (px) para que la página cargue rápido
# ----------------------------------------------------------------------

MESES = ["ENE", "FEB", "MAR", "ABR", "MAY", "JUN", "JUL", "AGO", "SEP", "OCT", "NOV", "DIC"]
NOMBRE = {"IDEAM": "IDEAM", "CHIRPS": "CHIRPS", "ERA5L": "ERA5-Land", "ERA5": "ERA5",
          "CHIRPS v2.0": "CHIRPS", "ERA5-Land": "ERA5-Land", "IDEAM (estaciones)": "IDEAM"}
os.makedirs(os.path.join(SALIDA, "img"), exist_ok=True)

## 2. Funciones de apoyo

In [14]:
def ruta(n):
    return os.path.join(RESULTADOS, n)


def existe(n):
    return bool(n) and os.path.exists(ruta(n))


def leer_csv(n, **kw):
    try:
        return pd.read_csv(ruta(n), **kw) if existe(n) else None
    except Exception as e:
        print(f"  No se pudo leer {n}: {e}")
        return None


def num(x, dec=0):
    # Formato en español: 37 267,5 (espacio fino para miles, coma decimal)
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "—"
    s = f"{float(x):,.{dec}f}"
    return s.replace(",", " ").replace(".", ",").replace("-", "−")


def esc(t):
    return html.escape(str(t))


def falta(nombre, punto):
    return (f'<div class="falta">No se encontró <code>{esc(nombre)}</code>. Corra el notebook del punto {punto} '
            f'en la carpeta de resultados y vuelva a generar el sitio.</div>')


def fig(nombre, pie, punto, clave=None):
    # Copia la imagen a sitio/img (reducida si es muy grande) y devuelve la figura con su explicación al lado
    if not existe(nombre):
        return falta(nombre, punto)
    destino = os.path.join(SALIDA, "img", os.path.basename(nombre))
    try:
        from PIL import Image
        im = Image.open(ruta(nombre))
        if im.width > ANCHO_MAX_IMG:
            im = im.resize((ANCHO_MAX_IMG, round(im.height * ANCHO_MAX_IMG / im.width)), Image.LANCZOS)
        im.save(destino, optimize=True)
    except Exception:
        shutil.copy(ruta(nombre), destino)
    alt = esc(re.sub("<[^>]+>", "", pie))
    t = TEXTOS.get(clave or os.path.basename(nombre), {})
    partes = ""
    for k, titulo in (("hizo", "Qué se hizo"), ("consid", "Consideraciones"), ("leer", "Cómo leerla")):
        v = t.get(k)
        if v:
            cuerpo = lista(v) if isinstance(v, (list, tuple)) else f"<p>{v}</p>"
            partes += f'<div class="fig-parte"><h4>{titulo}</h4>{cuerpo}</div>'
    return (f'<div class="fig-fila"><figure class="fig"><button class="zoom" type="button" aria-label="Ampliar: {alt}">'
            f'<img src="img/{esc(os.path.basename(nombre))}" alt="{alt}" loading="lazy"></button></figure>'
            f'<div class="fig-texto"><p class="fig-num">Figura @@N@@</p><h3 class="fig-tit">{pie}</h3>{partes}</div></div>')


def lista(items):
    items = [i for i in items if i]
    return "<ul>" + "".join(f"<li>{i}</li>" for i in items) + "</ul>" if items else ""


def tabla(df, columnas, titulo=None, nota=None):
    # columnas: {columna: (encabezado, decimales o None para texto)}
    if df is None or df.empty:
        return ""
    cab = "".join(f"<th>{esc(h)}</th>" for h, _ in columnas.values())
    filas = []
    for _, r in df.iterrows():
        celdas = []
        for c, (_, dec) in columnas.items():
            v = r.get(c, None)
            if dec is None or not isinstance(v, (int, float, np.integer, np.floating)) or isinstance(v, bool):
                celdas.append(f'<td class="t">{esc("—" if v is None or (isinstance(v, float) and not np.isfinite(v)) else v)}</td>')
            else:
                celdas.append(f'<td>{num(v, dec)}</td>')
        filas.append("<tr>" + "".join(celdas) + "</tr>")
    t = f'<div class="tabla"><table><thead><tr>{cab}</tr></thead><tbody>{"".join(filas)}</tbody></table></div>'
    return (f'<h3 class="sub">{titulo}</h3>' if titulo else "") + t + (f'<p class="nota">{nota}</p>' if nota else "")


GRAFICOS = []


def vals(a, d=2):
    # Lista de números redondeados para los gráficos (None donde no hay dato)
    s = pd.to_numeric(pd.Series(list(a), dtype="object"), errors="coerce").astype(float)
    return [None if not np.isfinite(v) else round(float(v), d) for v in s]


def grafico(gid, spec, titulo=None, nota=None):
    # Registra un gráfico interactivo (SVG dibujado en el navegador, sin librerías externas).
    # Los colores "--s1", "--ink", ... son variables CSS: el gráfico cambia solo con el tema claro/oscuro.
    spec.setdefault("hover", "x")
    GRAFICOS.append({"id": gid, "titulo": re.sub("<[^>]+>", "", titulo or ""), "spec": spec})
    return ((f'<h3 class="sub">{titulo}</h3>' if titulo else "") + f'<div class="chart" id="{gid}"></div>'
            + (f'<p class="nota">{nota}</p>' if nota else ""))


def picos(ciclo, prominencia=0.10):
    from scipy.signal import find_peaks
    x = np.asarray(ciclo, float)
    p, _ = find_peaks(np.r_[x, x, x], prominence=prominencia * x.mean())
    return sorted({int(i - 12) for i in p if 12 <= i < 24})


def media_raster(n):
    # Media y extremos de un ráster (requiere rasterio); None si no se puede
    try:
        import rasterio
        with rasterio.open(ruta(n)) as s:
            a = s.read(1, masked=True)
        return float(a.mean()), float(a.min()), float(a.max())
    except Exception:
        return None


SECCIONES = []


def seccion(num_, titulo, resumen, figuras=(), bloques=(), extra=""):
    SECCIONES.append({"id": f"p{num_}", "num": num_, "titulo": titulo, "resumen": resumen,
                      "figuras": list(figuras), "bloques": list(bloques), "extra": extra})


def construir(f):
    # Ejecuta el constructor de una sección; si falla, la sección muestra el error en vez de romper todo el sitio
    try:
        f()
    except Exception as e:
        n = int(re.findall(r"\d+", f.__name__)[0])
        seccion(n, f.__doc__ or f.__name__, f'<span class="falta">No se pudo construir esta sección: {esc(e)}</span>')
        print(f"  Sección {n}: {e}")


# Tablas que usan varias secciones
BAL = leer_csv("tabla_balance_productos.csv", dtype={"codigo": str})
CUENCA = BAL[BAL.codigo == "CUENCA"].set_index("producto") if BAL is not None else None
CICLO = leer_csv("ciclo_anual_cuenca_productos.csv", index_col=0)
if CICLO is not None:
    CICLO = CICLO.drop("ANUAL", errors="ignore")      # la fila ANUAL es la suma; aquí solo se dejan los 12 meses
for n, t in {"tabla_balance_productos.csv": BAL, "ciclo_anual_cuenca_productos.csv": CICLO}.items():
    print(f"  {n}: {'ok' if t is not None else 'NO ENCONTRADO'}")

  tabla_balance_productos.csv: ok
  ciclo_anual_cuenca_productos.csv: ok


## 3. Explicación de cada figura
Este texto aparece al lado de cada figura en la página: **qué se hizo**, **consideraciones** y **cómo leerla**.
Puede editarlo aquí y volver a correr el notebook. Una entrada en forma de lista `[...]` se muestra con viñetas.
Si escribe el signo "menor que", póngalo como `&lt;`.

In [15]:
TEXTOS = {
    # ---------------------------- Puntos 1 y 2 ----------------------------
    "relieve": {
        "hizo": "Se recortó el DEM del curso (mdt900, celdas de ≈ 930 m) con el límite de la cuenca y se le agregó un "
                "sombreado de relieve para resaltar las cordilleras y los valles.",
        "leer": "La cuenca se alarga de sur a norte entre las cordilleras Occidental y Central, desde el Macizo Colombiano "
                "hasta el cañón del Cauca en Antioquia.",
    },
    "delimitacion": {
        "hizo": "Sobre el DEM del curso (mdt900, celdas de ≈ 930 m, EPSG:3116) se calcularon en QGIS la dirección y la "
                "acumulación de flujo con <code>r.watershed</code> de GRASS. Con <code>r.water.outlet</code> se obtuvieron "
                "todas las celdas que drenan al punto de cierre, en el sitio de presa de Hidroituango, y el resultado se "
                "convirtió en polígono. La red de drenaje se extrajo con <code>r.stream.extract</code> y el DEM se recortó "
                "con el límite de la cuenca.",
        "consid": [
            "El punto de cierre se ubicó sobre la celda del cauce con mayor acumulación cercana a la presa, para que no "
            "quedara en una ladera.",
            "Se usó el DEM sin remuestrear: su resolución basta para una cuenca de este tamaño.",
            "En las zonas planas del valle geográfico el trazado del río puede correrse alrededor de una celda (≈ 0,9 km).",
        ],
        "leer": "El color es la elevación. La línea negra es el límite de la cuenca, la azul el río principal y el "
                "triángulo el punto de cierre.",
    },
    # ---------------------------- Punto 3 ----------------------------
    "fig_P_anual_preliminar.png": {
        "hizo": "Se tomaron las normales 1981–2010 del IDEAM (hoja <i>Lluvia</i>); la precipitación anual de cada "
                "estación es la suma de sus 12 promedios mensuales. Se usaron las estaciones dentro de la cuenca y en una "
                "franja de 25 km alrededor de su límite, y se interpoló sobre la grilla del DEM (≈ 930 m) con el método de "
                "menor error en la validación cruzada, usando las 16 estaciones más cercanas a cada celda.",
        "consid": [
            "Se excluyeron las estaciones del vecindario que están en la vertiente del Pacífico: allí llueven "
            "5 000–12 000 mm/año, un régimen que no corresponde a la cuenca.",
            "Las estaciones con coordenadas repetidas se promediaron.",
            "Hay muchas estaciones en el valle y pocas en las cumbres, así que el efecto orográfico queda poco representado.",
        ],
        "leer": "Los puntos son las estaciones usadas y el color la lluvia estimada. Se distingue el valle geográfico, "
                "más seco, y el aumento de la lluvia hacia las cordilleras y el norte de la cuenca.",
    },
    "fig_validacion_cruzada.png": {
        "hizo": "Se sacó una estación a la vez, se estimó su lluvia con las demás y se comparó con el valor medido. Esto se "
                "repitió para todas las estaciones y para cada método: IDW con potencias 1, 2 y 3, y kriging ordinario con "
                "semivariogramas esférico, exponencial y gaussiano. Con los errores se calcularon RMSE, MAE, sesgo y r.",
        "consid": [
            "El método se eligió por el menor RMSE, que mide el error donde no hay estación, y no por preferencia.",
            "Las estaciones con mayor error pueden ser núcleos reales de lluvia o datos dudosos; se revisaron una a una.",
        ],
        "leer": "Cada punto es una estación. Sobre la línea 1:1 la estimación es exacta; la dispersión alrededor de ella "
                "es el error típico de la interpolación.",
    },
    "fig_semivariograma.png": {
        "hizo": "Se agruparon los pares de estaciones por la distancia que las separa y en cada grupo se calculó la "
                "semivarianza (la mitad de la diferencia cuadrática media de su lluvia). A esos puntos se ajustaron los "
                "modelos esférico, exponencial y gaussiano, con pepita, meseta y rango; son los que usa el kriging.",
        "consid": [
            "Una pepita alta indica mucha diferencia entre estaciones cercanas: efecto orográfico, microclimas o errores "
            "de medición.",
            "Si la curva sigue creciendo sin estabilizarse, hay una tendencia regional, del valle seco al norte húmedo.",
        ],
        "leer": "El eje horizontal es la distancia entre estaciones y el vertical cuánto difiere su lluvia. Mientras más "
                "rápido sube la curva, menos se parecen las estaciones vecinas.",
    },
    # ---------------------------- Punto 4 ----------------------------
    "fig_T_vs_Z_conjunta.png": {
        "hizo": "Se tomaron los promedios anuales 1981–2010 de temperatura máxima, mínima y media (promedio de los 12 "
                "meses, no los extremos anuales) de las estaciones dentro de la cuenca y en su vecindario. Para cada "
                "variable se ajustó por mínimos cuadrados la recta T = a + bZ, de donde T₀ = a y α = −b.",
        "consid": [
            "Hay muchas menos estaciones de temperatura que de lluvia: solo las climatológicas, sinópticas y "
            "agrometeorológicas la miden.",
            "Se revisaron las estaciones que se alejan de la recta más de 2,5 veces el error típico (valles abrigados, "
            "laderas expuestas, ciudades).",
        ],
        "leer": "Cada punto es una estación y cada línea la regresión de su variable. La pendiente es el gradiente α y el "
                "R² indica cuánto de la variación explica la elevación.",
    },
    # ---------------------------- Punto 5 ----------------------------
    "fig_mapas_T_misma_escala.png": {
        "hizo": "Con álgebra de mapas se aplicó a cada celda del DEM de la cuenca la ecuación T = T₀ − αZ de cada "
                "variable (punto 4). Como el DEM ya está recortado, los mapas quedan limitados a la cuenca.",
        "consid": [
            "Los tres mapas tienen la misma escala de color para comparar magnitudes directamente.",
            "Es una relación solo con la elevación: no representa valles secos más cálidos, la exposición de las laderas "
            "ni las inversiones térmicas.",
            "En celdas más altas que la estación más alta, o más bajas que la más baja, la ecuación se extrapola.",
        ],
        "leer": "Los mapas reproducen el relieve: más cálido en el fondo del valle y del cañón, más frío en las cumbres "
                "de la cordillera Central y en el Macizo.",
    },
    "fig_pisos_termicos.png": {
        "hizo": "La Tmed de cada celda se clasificó en los pisos térmicos de Caldas: cálido (> 24 °C), templado (18–24), "
                "frío (12–18), páramo (6–12) y nival (&lt; 6).",
        "consid": [
            "Los límites de elevación de cada piso salen de la regresión de esta cuenca, no de valores genéricos.",
            "Es una clasificación solo por temperatura; no tiene en cuenta humedad ni vegetación.",
        ],
        "leer": "Muestra qué parte de la cuenca está en cada piso y dónde están los sectores más fríos (cumbres) y más "
                "cálidos (fondos de valle).",
    },
    "fig_coherencia_mapa_estaciones.png": {
        "hizo": "Se extrajo el valor del mapa en la celda de cada estación y se comparó con la temperatura medida. También "
                "se comparó la elevación del DEM en esa celda con la de la estación, y se verificó que Tmax > Tmed > Tmin "
                "en todas las celdas.",
        "consid": [
            "Parte de la diferencia no es error del método: la celda de ≈ 930 m tiene una elevación promedio distinta "
            "de la del punto donde está la estación.",
        ],
        "leer": "Puntos cerca de la línea 1:1 indican que el mapa reproduce las estaciones; las diferencias grandes "
                "suelen coincidir con diferencias de elevación entre la celda y la estación.",
    },
    # ---------------------------- Punto 6 ----------------------------
    "fig_mapa_ETP.png": {
        "hizo": "Para cada celda y cada mes se calculó ETP<sub>d</sub> = 0,0023·R<sub>a</sub>*·(Tmed + 17,8)·√(Tmax − Tmin), "
                "con las temperaturas del punto 5 y la radiación extraterrestre de ese mes en la latitud de la celda "
                "(Anexo 3). La ETP mensual es la diaria por los días del mes, y la anual es la suma de los 12 meses.",
        "consid": [
            "La Tmed se tomó como (Tmax + Tmin)/2, como indica el enunciado.",
            "R<sub>a</sub> se calculó con la latitud de cada celda porque la cuenca va de unos 2° a 7° N.",
            "Hargreaves no usa humedad ni viento; la nubosidad entra solo a través de la amplitud térmica Tmax − Tmin.",
        ],
        "leer": "La ETP es mayor en las zonas bajas y cálidas y disminuye con la altura, siguiendo el patrón de la "
                "temperatura.",
    },
    "fig_Ra_mensual.png": {
        "hizo": "Se calculó R<sub>a</sub> para el día representativo de cada mes (días julianos 15, 45, 74… 349) con las "
                "ecuaciones del Anexo 3: distancia Tierra–Sol, declinación solar y ángulo de puesta del sol. Se convirtió "
                "a mm/día con R<sub>a</sub>* = 0,408·R<sub>a</sub>.",
        "consid": ["Se muestran las latitudes extremas y la media de la cuenca para ver cuánto cambia R<sub>a</sub> de sur "
                   "a norte."],
        "leer": "Cerca del ecuador R<sub>a</sub> cambia poco en el año, con máximos cerca de los equinoccios (marzo y "
                "septiembre). Por eso el ciclo anual de la ETP es suave.",
    },
    "fig_ETP_analisis.png": {
        "hizo": "Se comparó la ETP anual de cada celda con su temperatura media y se calculó el promedio de la cuenca "
                "para cada mes.",
        "consid": ["La versión simplificada (ETP diaria con el R<sub>a</sub> medio del año × 365) da prácticamente lo "
                   "mismo que la suma de los 12 meses."],
        "leer": "La relación casi lineal con la temperatura muestra que la elevación controla la ETP; el ciclo mensual "
                "sigue sobre todo al de R<sub>a</sub>.",
    },
    # ---------------------------- Punto 7 ----------------------------
    "fig_mapas_phi_ETR_R.png": {
        "hizo": "En cada celda se calculó el índice de aridez φ = ETP/P, con la ETP del punto 6 y la P del IDEAM del "
                "punto 3, y la ETR con la curva de Budyko: ETR = P·[φ·tanh(1/φ)·(1 − e<sup>−φ</sup>)]<sup>½</sup>. "
                "La escorrentía de largo plazo es R = P − ETR.",
        "consid": [
            "En el enunciado la fórmula aparece como «tanh tanh»; se usó una sola tangente hiperbólica, como en Budyko (1974).",
            "Budyko es de largo plazo: supone que el almacenamiento no cambia (ΔS ≈ 0) y no tiene parámetros de suelo "
            "ni de vegetación.",
        ],
        "leer": "φ &lt; 1 es zona húmeda, donde la ETR la limita la energía; φ > 1 es zona seca, donde la limita el agua. "
                "El mapa de R muestra de dónde sale el agua que llega al río.",
    },
    "fig_espacio_budyko.png": {
        "hizo": "Cada celda se ubicó según su índice de aridez φ (eje horizontal) y su ETR/P (eje vertical), junto a la "
                "curva de Budyko y los límites de energía (ETR = ETP) y de agua (ETR = P).",
        "consid": ["Todas las celdas caen sobre la curva porque la ETR se calculó con ella. La figura muestra en qué "
                   "régimen está la cuenca; la validación con caudales se hace en el punto 9."],
        "leer": "Las celdas de la izquierda (φ pequeño) evaporan una fracción pequeña de la lluvia y aportan más "
                "escorrentía; las de la derecha, del valle seco, evaporan la mayor parte.",
    },
    # ---------------------------- Punto 8 ----------------------------
    "fig_estaciones_areas_drenaje.png": {
        "hizo": "Las coordenadas de cada estación (Catálogo Nacional de Estaciones del IDEAM) se llevaron a la celda del "
                "río en el DEM: la de mayor acumulación cercana, siempre que el rendimiento q = Q/A quedara entre 8 y "
                "150 L/s/km². Desde esa celda se delimitó el área de drenaje con la dirección de flujo D8 del mismo DEM.",
        "consid": [
            "Si no había una celda válida a menos de 2 km, la búsqueda se amplió de 2 en 2 km hasta 10 km: en el valle "
            "geográfico el río del DEM puede quedar a varios km del real.",
            "La Pintada se delimitó antes de la desembocadura del río Arma.",
            "Puerto Valdivia queda aguas abajo de la presa; parte de su área sale de la cuenca de estudio.",
        ],
        "leer": "Cada color es el área que drena a una estación; las áreas se anidan de aguas arriba a aguas abajo. "
                "La etiqueta da el área y el caudal medio.",
    },
    "fig_disponibilidad_caudales.png": {
        "hizo": "Se descargó el caudal medio diario de cada estación (datos abiertos del IDEAM). Un mes se consideró "
                "válido con datos en al menos el 80 % de sus días y un año completo con sus 12 meses válidos. Se marcaron "
                "como aptas las estaciones con 15 o más años completos en 1981–2010.",
        "consid": [
            "El periodo 1981–2010 coincide con el de la climatología de lluvia y temperatura.",
            "Los faltantes no se rellenaron.",
        ],
        "leer": "Cada barra muestra los meses con dato y la franja gris es 1981–2010. Las estaciones en rojo no tienen "
                "suficientes años completos y no se usan.",
    },
    "fig_Q_anual_estaciones.png": {
        "hizo": "Para cada año completo se calculó el caudal medio anual (promedio de sus 12 meses). El caudal medio "
                "multianual de la estación es el promedio de esos años.",
        "consid": ["Se recalcularon las medias exigiendo 70 %, 90 % y 95 % de días por mes: cambian muy poco, así que el "
                   "resultado no depende del criterio de faltantes."],
        "leer": "Permite ver la variabilidad entre años, por ejemplo años húmedos de La Niña y secos de El Niño, y "
                "detectar saltos sospechosos en una estación.",
    },
    # ---------------------------- Puntos 10 y 11 ----------------------------
    "fig_mapas_P_productos.png": {
        "hizo": "Se descargaron los productos para 1981–2010 en una caja que cubre la cuenca con 0,5° de margen: CHIRPS "
                "v2.0 de la IRI Data Library (mm/mes), y ERA5 y ERA5-Land del Copernicus Climate Data Store (promedios "
                "mensuales en m/día, convertidos con P = tp × 1000 × días del mes). En la grilla de cada producto se "
                "promedió cada mes sobre los 30 años; la P anual es la suma de los 12 promedios.",
        "consid": [
            "Para mostrar los píxeles reales de cada producto, los mapas se pasan a la grilla del DEM por vecino más "
            "cercano; para los cálculos (medias, balance) se usa interpolación bilineal.",
            "La escala de color es por clases y es la misma en los cuatro mapas: detalla el rango de 1 000 a 3 500 mm/año "
            "y deja ver los valores altos de ERA5 y ERA5-Land.",
        ],
        "leer": "Compara magnitud y patrón: si cada producto reproduce el valle geográfico seco y las zonas más "
                "lluviosas, y cuánto suaviza la resolución gruesa.",
    },
    "fig_mapa_P_CHIRPS.png": {
        "hizo": "CHIRPS combina estimaciones de satélite (infrarrojo) con estaciones y una climatología de alta "
                "resolución. Se usó la versión mensual a 0,05° (≈ 5,5 km).",
        "consid": ["Incluye estaciones, muchas del IDEAM, así que tiende a parecerse al IDEAM en magnitud y no es del "
                   "todo independiente."],
        "leer": "Cada cuadro es un píxel de CHIRPS. La escala es la misma de la figura comparativa.",
    },
    "fig_mapa_P_ERA5L.png": {
        "hizo": "ERA5-Land es un reanálisis de superficie a 0,1° (≈ 11 km) forzado con la atmósfera de ERA5. Se usaron "
                "los promedios mensuales de precipitación total.",
        "consid": ["Su precipitación proviene de ERA5, así que ERA5 y ERA5-Land no son estimaciones independientes; por "
                   "eso sus valores son tan parecidos."],
        "leer": "Cada cuadro es un píxel de ERA5-Land. La escala es la misma de la figura comparativa.",
    },
    "fig_mapa_P_ERA5.png": {
        "hizo": "ERA5 es el reanálisis atmosférico global del ECMWF a 0,25° (≈ 28 km). Su precipitación la calcula el "
                "modelo; no es una medición.",
        "consid": [
            "Con píxeles de ≈ 28 km la cuenca queda cubierta por pocos píxeles y se pierde el contraste entre el valle "
            "y las laderas.",
            "Los reanálisis tienden a sobrestimar la lluvia sobre el relieve andino.",
        ],
        "leer": "Cada cuadro es un píxel de ERA5. La escala es la misma de la figura comparativa.",
    },
    "fig_verificacion_productos.png": {
        "hizo": "Se dibujó la precipitación anual de cada producto en toda la zona descargada, en grados, con el límite de "
                "la cuenca y seis lugares de referencia: Buenaventura, Quibdó, Cali, Popayán, Manizales y Medellín.",
        "consid": ["Sirve para comprobar que cada archivo se leyó en la ubicación correcta: la zona más lluviosa de "
                   "Colombia es la costa Pacífica y el valle del Cauca es seco."],
        "leer": "Si el máximo de cada producto está hacia el Pacífico, la lectura es correcta y las diferencias dentro de "
                "la cuenca son del propio producto.",
    },
    # ---------------------------- Puntos 9 y 12 ----------------------------
    "fig_tabla12_cierre.png": {
        "hizo": "Para cada fuente de lluvia se calculó la ETR de Budyko celda a celda con su propia P y la misma ETP de "
                "Hargreaves. Luego se promediaron P, ETR y R = P − ETR en el área de drenaje de la estación válida más "
                "cercana a la presa, y el Q estimado se comparó con el caudal medido en esa estación.",
        "consid": [
            "No hay estación exactamente en la presa; la tabla de esta sección indica cuál se usó como cierre.",
            "La banda gris es ± 20 % del caudal observado, el error que se considera aceptable.",
        ],
        "leer": "A la izquierda, cuánta lluvia aporta cada producto y cuánta se evapora; a la derecha, si el agua que "
                "queda coincide con la que mide el río.",
    },
    "fig_evaluacion_productos.png": {
        "hizo": "Para cada estación y producto se buscó el factor que, aplicado a la lluvia del producto en cada celda, "
                "hace que el balance dé exactamente el caudal observado: esa es la P necesaria. También se comparó la "
                "escorrentía estimada con la observada en todas las estaciones.",
        "consid": [
            "Así la lluvia de cada producto se evalúa contra el caudal medido, y no solo contra el IDEAM, que también "
            "tiene error.",
            "La estrella marca la estación del cierre.",
        ],
        "leer": "Puntos sobre la línea 1:1 indican lluvia compatible con el caudal; por encima, el producto llueve de "
                "más; por debajo, de menos.",
    },
    "fig_balance_productos.png": {
        "hizo": "Se repitió el balance en el área de drenaje de cada estación con cada producto y se calculó el error "
                "(Q estimado − Q observado)/Q observado.",
        "consid": [
            "Solo se usan las estaciones con al menos el 95 % de su área de drenaje dentro de los mapas.",
            "Un error con el mismo signo en todas las estaciones indica un sesgo del producto; si cambia de signo, "
            "puede haber compensación de errores.",
        ],
        "leer": "Barras cerca de cero, dentro de las líneas punteadas de ± 20 %, indican que el producto cierra el balance "
                "en esa estación.",
    },
    "fig_budyko_estaciones.png": {
        "hizo": "Para cada estación y producto se calculó la ETR «observada» como la P del producto menos el caudal "
                "medido (en mm/año), y se ubicó en el espacio de Budyko junto a la curva teórica.",
        "consid": ["Si un punto queda por encima de la línea ETR = ETP o por debajo de 0, el balance no puede cerrar con "
                   "esa lluvia."],
        "leer": "Sobre la curva, el producto es coherente con el caudal; por encima, llueve de más o el caudal está "
                "disminuido; por debajo, llueve de menos.",
    },
    "fig_mapas_R_productos.png": {
        "hizo": "Con la P de cada producto y su ETR de Budyko se calculó la escorrentía de largo plazo R = P − ETR en "
                "cada celda.",
        "consid": ["La ETP es la misma para todos, así que las diferencias entre los mapas vienen solo de la "
                   "precipitación."],
        "leer": "Muestra dónde genera cada producto el agua que llega al río; compárelo con los rendimientos observados "
                "del punto 8.",
    },
    # ---------------------------- Punto 13 ----------------------------
    "fig_P_mensual_12mapas_IDEAM.png": {
        "hizo": "Se interpolaron los 12 promedios mensuales 1981–2010 de las estaciones con el mismo método y la misma "
                "grilla del mapa anual (punto 3). El ciclo anual de la cuenca es el promedio de cada mapa sobre la cuenca.",
        "consid": [
            "La suma de los 12 mapas coincide con el mapa anual porque la P anual se calculó como la suma de los meses.",
            "Los 12 mapas tienen la misma escala de color.",
        ],
        "leer": "Muestra en qué meses y en qué sectores llueve más, y cómo se desplazan las temporadas lluviosas.",
    },
    "fig_variacion_espacial_ciclo.png": {
        "hizo": "Con los 12 mapas mensuales se calculó en cada celda: el mes de máxima lluvia; el número de temporadas "
                "lluviosas, contando los máximos del ciclo que sobresalen al menos 10 % de la media mensual; y el índice "
                "de estacionalidad de Walsh y Lawler (1981).",
        "consid": ["Un índice de estacionalidad menor de 0,2 indica lluvia repartida en el año; mayor de 0,4, una "
                   "estación seca marcada."],
        "leer": "Muestra dónde cambia el régimen dentro de la cuenca, por ejemplo si hacia el norte se debilita el periodo "
                "seco de mitad de año.",
    },
    "fig_ciclo_por_franjas.png": {
        "hizo": "La cuenca se dividió en tres franjas de latitud (sur, centro y norte) y se calculó el ciclo anual "
                "promedio de cada una.",
        "consid": ["Las franjas son una simplificación; los mapas de la figura anterior dan el detalle."],
        "leer": "Compara el ciclo de sur a norte: magnitud, meses de máximo y profundidad de los periodos secos.",
    },
    # ---------------------------- Punto 14 ----------------------------
    "fig_taylor_ciclo_anual.png": {
        "hizo": "Para el ciclo anual de la cuenca de cada producto (12 valores) se calcularon la desviación estándar, la "
                "correlación con el ciclo del IDEAM y el error cuadrático medio centrado, y se ubicaron en el diagrama "
                "de Taylor (2001).",
        "consid": [
            "El diagrama no muestra el sesgo, por ejemplo un producto que llueve de más todo el año; por eso va "
            "acompañado de la tabla de métricas.",
            "La referencia es el IDEAM, que también tiene incertidumbre.",
        ],
        "leer": "La distancia al origen es la amplitud del ciclo, el ángulo la correlación y la distancia a la estrella "
                "el error. El mejor producto es el más cercano a la estrella.",
    },
    "fig_diferencias_espaciales.png": {
        "hizo": "Celda por celda se calculó la diferencia relativa (P producto − P IDEAM)/P IDEAM, con todos los mapas "
                "en la grilla del DEM.",
        "consid": ["El mapa del IDEAM es una interpolación, no la verdad: las diferencias grandes en las cumbres pueden ser "
                   "error del IDEAM y no del producto."],
        "leer": "Azul: el producto llueve más que el IDEAM; rojo: menos. La escala va de −80 % a +80 %.",
    },
    "fig_punto_pixel.png": {
        "hizo": "Se extrajo el valor de cada producto en el píxel de cada estación dentro de la cuenca y se comparó con la "
                "P medida. Para el IDEAM se usó el valor de la validación cruzada, estimado sin esa estación, para que "
                "la comparación sea justa.",
        "consid": ["Un píxel de 28 km (ERA5) no puede reproducir un valor puntual; parte del error es de escala, no del "
                   "producto."],
        "leer": "Puntos cerca de la línea 1:1 indican buen acuerdo; el título de cada panel da el RMSE y la correlación.",
    },
}

## 4. Contenido de cada sección

In [16]:
def s1():
    "Selección de la cuenca"
    area = CUENCA.loc["IDEAM", "area_km2"] if CUENCA is not None and "IDEAM" in CUENCA.index else None
    dem = media_raster("DEM_cuenca.tif")
    resumen = (f"Cuenca del río Cauca hasta el sitio de presa de <b>Hidroituango</b> ({num(CIERRE[0], 2)}° N, "
               f"{num(abs(CIERRE[1]), 2)}° O), en Antioquia" + (f": <b>{num(area)} km²</b>." if area else "."))
    bloques = [("Por qué esta cuenca", lista([
        "Es una de las cuencas preseleccionadas en el curso y la principal del occidente andino colombiano.",
        "Va desde el Macizo Colombiano hasta el cañón de Antioquia: cubre casi todos los pisos térmicos y regímenes de lluvia.",
        "El cierre en Hidroituango permite un balance con el periodo 1981–2010, anterior al embalse, sin su efecto en los caudales.",
    ]))]
    if dem:
        bloques.append(("En cifras", lista([f"Elevación de {num(dem[1])} a {num(dem[2])} m s. n. m. (media {num(dem[0])} m)",
                                            f"Área: {num(area)} km²" if area else None])))
    # Si hay mapa de QGIS para la sección 2, el DEM va aquí; si no, el DEM se muestra en la sección 2
    figs = [fig("fig_dem_cuenca.png", "Modelo digital de elevación de la cuenca del Cauca hasta Hidroituango.", 2, "relieve")] \
        if MAPA_DELIMITACION and existe("fig_dem_cuenca.png") else []
    seccion(1, s1.__doc__, resumen, figs, bloques)


def s2():
    "Delimitación"
    area = CUENCA.loc["IDEAM", "area_km2"] if CUENCA is not None and "IDEAM" in CUENCA.index else None
    if MAPA_DELIMITACION:
        figura = fig(MAPA_DELIMITACION, "Cuenca delimitada: límite, red de drenaje, río principal, punto de cierre y DEM.", 2,
                     "delimitacion")
    elif existe("fig_dem_cuenca.png"):
        figura = fig("fig_dem_cuenca.png", "Cuenca delimitada sobre el DEM, con el punto de cierre.", 2, "delimitacion")
    else:
        figura = ('<div class="falta">Falta el mapa de la cuenca. Exporte el mapa de QGIS como imagen (Proyecto → Importar/Exportar → '
                  'Exportar mapa a imagen), guárdelo en la carpeta de resultados y escriba su nombre en <code>MAPA_DELIMITACION</code>. '
                  'También sirve dejar <code>DEM_cuenca.tif</code> en la carpeta: este notebook dibuja el mapa solo.</div>')
    seccion(2, s2.__doc__,
            "Delimitación con el DEM del curso (mdt900, celdas de ≈ 930 m) en MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)"
            + (f"; área resultante de <b>{num(area)} km²</b>." if area else "."),
            [figura],
            [("Metodología", lista([
                "<b>Dirección y acumulación de flujo</b> con <code>r.watershed</code> de GRASS en QGIS (flujo D8; las depresiones se resuelven con el algoritmo de menor costo).",
                "<b>Punto de cierre</b> en la celda de máxima acumulación del cauce más cercana a la presa de Hidroituango.",
                "<b>Cuenca</b> con <code>r.water.outlet</code>; <b>red de drenaje</b> con <code>r.stream.extract</code> (umbral de 100 celdas ≈ 87 km²).",
            ])),
             ("Decisiones", lista([
                 "<b>DEM sin remuestrear.</b> La resolución de ≈ 930 m es suficiente para una cuenca de este tamaño y evita inventar detalle.",
                 "<b>Cierre en la presa y no en la confluencia con el Magdalena.</b> La zona plana de La Mojana no se puede delimitar bien con un DEM.",
             ]))])


def s3():
    "Precipitación"
    est = leer_csv("estaciones_P_seleccionadas.csv")
    cv = leer_csv("validacion_cruzada_metodos.csv")
    P = CICLO["IDEAM"].sum() if CICLO is not None and "IDEAM" in CICLO else None
    n_d = int((est.ubicacion == "dentro").sum()) if est is not None else None
    n_v = int((est.ubicacion == "vecindario").sum()) if est is not None else None
    if cv is not None:
        cv = cv.sort_values("RMSE_mm").reset_index(drop=True)
    mejor = cv.iloc[0] if cv is not None else None
    resumen = ((f"{n_d} estaciones dentro de la cuenca y {n_v} en su vecindario. " if est is not None else "")
               + (f"Método elegido por validación cruzada: <b>{esc(mejor.metodo)}</b> (RMSE = {num(mejor.RMSE_mm)} mm/año, "
                  f"r = {num(mejor.r, 2)}). " if mejor is not None else "")
               + (f"Precipitación media de la cuenca: <b>{num(P)} mm/año</b>." if P else ""))
    extra = tabla(cv, {"metodo": ("Método", None), "RMSE_mm": ("RMSE (mm/año)", 0), "MAE_mm": ("MAE (mm/año)", 0),
                       "Sesgo_mm": ("Sesgo (mm/año)", 0), "r": ("r", 2)},
                  "Validación cruzada dejando una estación por fuera",
                  "El método con menor RMSE es el que mejor estima la lluvia donde no hay estación.")
    seccion(3, s3.__doc__, resumen,
            [fig("fig_P_anual_preliminar.png", "Precipitación media anual 1981–2010 interpolada, con las estaciones usadas.", 3),
             fig("fig_validacion_cruzada.png", "Precipitación observada vs. estimada en la validación cruzada.", 3),
             fig("fig_semivariograma.png", "Semivariograma empírico y modelos ajustados.", 3)],
            [("Metodología", lista([
                "Normales climatológicas 1981–2010 del IDEAM (hoja <i>Lluvia</i>); la precipitación anual es la suma de los 12 promedios mensuales.",
                "Estaciones <b>dentro</b> de la cuenca y en un <b>vecindario de 25 km</b> alrededor del límite.",
                "Comparación de IDW (potencias 1 a 3) y kriging ordinario (modelos esférico, exponencial y gaussiano) por <b>validación cruzada</b>.",
                "Interpolación sobre la misma grilla del DEM, para combinar el mapa con los de temperatura y ETP.",
            ])),
             ("Decisiones", lista([
                 "<b>Se excluye la vertiente del Pacífico.</b> Al otro lado de la cordillera Occidental llueven 5 000–12 000 mm/año, un régimen que no representa la cuenca.",
                 "<b>El método se elige por datos.</b> Se usa el de menor RMSE de validación cruzada, no una preferencia a priori.",
                 "<b>Limitación.</b> Ningún método representa explícitamente el efecto orográfico, y en las zonas altas hay pocas estaciones.",
             ]))], extra)


def s4():
    "Temperatura y elevación"
    reg = leer_csv("resultados_regresion_T.csv")
    est = leer_csv("estaciones_T_seleccionadas.csv")
    if reg is None:
        seccion(4, s4.__doc__, falta("resultados_regresion_T.csv", 4)); return
    r = reg.set_index("clave")
    resumen = (f"Gradiente de la temperatura media: <b>α = {num(r.loc['tmed', 'alpha_C_por_km'], 2)} °C/km</b> "
               f"(R² = {num(r.loc['tmed', 'R2'], 3)}, n = {int(r.loc['tmed', 'n_estaciones'])}). "
               f"Tmax: {num(r.loc['tmax', 'alpha_C_por_km'], 2)} °C/km; Tmin: {num(r.loc['tmin', 'alpha_C_por_km'], 2)} °C/km.")
    series = []
    color = {"tmax": "--s2", "tmed": "--s3", "tmin": "--s1"}
    nom = {"tmax": "Tmax", "tmed": "Tmed", "tmin": "Tmin"}
    zz = [0, round(float(est.ELEV.max()) * 1.05)] if est is not None else [0, 4500]
    for v in ("tmax", "tmed", "tmin"):
        if est is not None:
            d = est[["NOMBRE", "ELEV", v]].dropna()
            series.append({"name": nom[v], "kind": "dots", "color": color[v], "x": vals(d.ELEV, 0), "y": vals(d[v], 2),
                           "labels": d.NOMBRE.astype(str).tolist(), "dec": 1, "unit": " °C", "legend": False})
        T0, a = r.loc[v, "T0_C"], r.loc[v, "alpha_C_por_m"]
        series.append({"name": nom[v], "kind": "line", "legendKind": "linedots", "color": color[v], "x": zz,
                       "y": vals([T0 - a * z for z in zz], 2), "hover": False})
    extra = (grafico("g4", {"x": {"type": "lin", "title": "Elevación (m s. n. m.)", "name": "Z", "unit": " m", "dec": 0},
                            "y": {"type": "lin", "title": "Temperatura (°C)"}, "series": series, "hover": "point",
                            "labelTitle": "Estación", "height": 440},
                     "Temperatura vs. elevación", "Puntos: estaciones; líneas: regresiones T = T₀ − αZ.")
             + tabla(reg, {"variable": ("Variable", None), "ecuacion": ("Ecuación (Z en m)", None),
                           "alpha_C_por_km": ("α (°C/km)", 2), "IC95_alpha_C_por_km": ("± IC 95 %", 2), "R2": ("R²", 3),
                           "RMSE_C": ("Error típico (°C)", 2), "n_estaciones": ("n", 0)}, "Regresiones T = T₀ − αZ"))
    seccion(4, s4.__doc__, resumen,
            [fig("fig_T_vs_Z_conjunta.png", "Temperatura vs. elevación para Tmax, Tmed y Tmin.", 4)],
            [("Metodología", lista([
                "Promedios anuales 1981–2010 de Tmax, Tmin y Tmed (hoja <i>Temperatura</i> del IDEAM), con estaciones dentro de la cuenca y en su vecindario.",
                "Regresión lineal T = a + bZ por mínimos cuadrados, con T₀ = a y α = −b (Anexo 1).",
            ])),
             ("Interpretación", lista([
                 f"<b>α(Tmed) ≈ {num(r.loc['tmed', 'alpha_C_por_km'], 1)} °C/km.</b> Queda entre el gradiente adiabático saturado (≈ 4–6 °C/km) y el seco (9,8 °C/km), cerca de la atmósfera estándar (6,5 °C/km): es una atmósfera húmeda.",
                 ("<b>α(Tmax) > α(Tmin).</b> La amplitud térmica diaria disminuye con la altura." if r.loc['tmax', 'alpha_C_por_km'] > r.loc['tmin', 'alpha_C_por_km']
                  else "<b>α(Tmin) ≥ α(Tmax).</b> La amplitud térmica diaria no disminuye con la altura."),
                 f"<b>T₀ ({num(r.loc['tmed', 'T0_C'], 1)} °C) es una extrapolación.</b> No hay estaciones cerca del nivel del mar; no es una temperatura observada.",
                 f"<b>R² alto (≥ {num(reg.R2.min(), 2)}).</b> La elevación explica casi toda la variación espacial; el resto son efectos locales (valles, laderas, ciudades).",
             ]))], extra)


def s5():
    "Mapas de temperatura"
    t = media_raster("Tmed.tif")
    pisos = None
    try:
        import rasterio
        with rasterio.open(ruta("pisos_termicos.tif")) as s:
            a = s.read(1, masked=True)
        nombres = ["Nival / glacial", "Páramo", "Frío", "Templado", "Cálido"]
        pisos = pd.DataFrame({"piso": nombres, "pct": [100 * float((a == i).sum()) / a.count() for i in range(5)]})
    except Exception:
        pass
    resumen = ("Campos de Tmax, Tmin y Tmed por álgebra de mapas, T = T₀ − αZ, celda a celda sobre el DEM"
               + (f": la Tmed va de <b>{num(t[1], 1)} °C</b> en las cumbres a <b>{num(t[2], 1)} °C</b> en el fondo del cañón "
                  f"(media {num(t[0], 1)} °C)." if t else "."))
    bloques = [("Metodología", lista([
        "Se aplica a cada celda la ecuación de la sección 4 con su elevación Z.",
        "La Tmed se clasifica en pisos térmicos de Caldas: cálido > 24 °C, templado 18–24, frío 12–18, páramo 6–12, nival < 6.",
        "Coherencia: Tmax > Tmed > Tmin en todas las celdas, y comparación del mapa con las estaciones.",
    ])),
        ("Análisis", lista([
            "<b>Efecto de la elevación.</b> La temperatura disminuye linealmente con la altura, así que el mapa reproduce el relieve.",
            "<b>Sectores más cálidos.</b> El fondo del cañón del Cauca cerca de Hidroituango y el valle geográfico del Cauca.",
            "<b>Sectores más fríos.</b> Las cumbres de la cordillera Central (zona de los nevados) y el Macizo Colombiano.",
            "<b>Limitaciones.</b> El método no representa valles secos más cálidos, laderas expuestas ni inversiones térmicas; las celdas de ≈ 930 m suavizan cumbres y fondos de valle.",
        ]))]
    if pisos is not None:
        bloques.append(("Pisos térmicos", lista([f"{p.piso}: {num(p.pct, 1)} % del área" for p in pisos.itertuples()])))
    seccion(5, s5.__doc__, resumen,
            [fig("fig_mapas_T_misma_escala.png", "Tmax, Tmed y Tmin con la misma escala de color.", 5),
             fig("fig_pisos_termicos.png", "Pisos térmicos según la Tmed estimada.", 5),
             fig("fig_coherencia_mapa_estaciones.png", "Coherencia: mapa vs. estaciones y efecto de la resolución del DEM.", 5)],
            bloques)


def s6():
    "Evapotranspiración potencial"
    etp = CUENCA.loc["IDEAM", "ETP_mm"] if CUENCA is not None and "IDEAM" in CUENCA.index else None
    resumen = ("ETP media multianual con <b>Hargreaves-Samani</b>, usando la radiación extraterrestre calculada para la latitud de cada celda"
               + (f": <b>{num(etp)} mm/año</b> en promedio sobre la cuenca." if etp else "."))
    seccion(6, s6.__doc__, resumen,
            [fig("fig_mapa_ETP.png", "Evapotranspiración potencial media anual (Hargreaves-Samani).", 6),
             fig("fig_Ra_mensual.png", "Radiación extraterrestre equivalente (mm/día) por mes y latitud.", 6),
             fig("fig_ETP_analisis.png", "ETP vs. temperatura media y ciclo anual de la ETP.", 6)],
            [("Metodología", lista([
                "ETP<sub>d</sub> = 0,0023·R<sub>a</sub>*·(T<sub>med</sub> + 17,8)·√(T<sub>max</sub> − T<sub>min</sub>), con T<sub>med</sub> = (T<sub>max</sub> + T<sub>min</sub>)/2.",
                "R<sub>a</sub> (Anexo 3) para el día representativo de cada mes y la <b>latitud de cada celda</b>; R<sub>a</sub>* = 0,408·R<sub>a</sub> en mm/día.",
                "ETP<sub>m</sub> = ETP<sub>d</sub>·N<sub>d</sub>, y la anual es la suma de los 12 meses. Da prácticamente lo mismo que ETP<sub>d</sub> × 365.",
            ])),
             ("Análisis", lista([
                 "La ETP es mayor en las zonas bajas y cálidas y disminuye con la altura, siguiendo el mapa de Tmed.",
                 "Cerca del ecuador R<sub>a</sub> cambia poco durante el año, así que el ciclo anual de la ETP es suave.",
                 "<b>Limitación.</b> Hargreaves no usa humedad ni viento; la nubosidad solo entra a través de Tmax − Tmin.",
             ]))])


def s7():
    "Evapotranspiración real"
    c = CUENCA.loc["IDEAM"] if CUENCA is not None and "IDEAM" in CUENCA.index else None
    resumen = ("ETR de largo plazo con la curva de <b>Budyko</b>, ETR = P·[φ·tanh(1/φ)·(1 − e<sup>−φ</sup>)]<sup>½</sup> con φ = ETP/P"
               + (f": φ = {num(c.phi, 2)}, <b>ETR = {num(c.ETR_mm)} mm/año</b> (ETR/P = {num(c.ETR_mm / c.P_mm, 2)}) y "
                  f"escorrentía P − ETR = {num(c.Q_est_mm)} mm/año." if c is not None else "."))
    seccion(7, s7.__doc__, resumen,
            [fig("fig_mapas_phi_ETR_R.png", "Índice de aridez φ, ETR de Budyko y escorrentía R = P − ETR.", 7),
             fig("fig_espacio_budyko.png", "Celdas de la cuenca en el espacio de Budyko.", 7)],
            [("Metodología", lista([
                "φ = ETP/P celda a celda, con la P del IDEAM (sección 3) y la ETP de Hargreaves (sección 6).",
                "En el enunciado la fórmula aparece como «tanh tanh»; se usó <b>una sola tangente hiperbólica</b>, como en Budyko (1974).",
            ])),
             ("Análisis", lista([
                 "<b>φ < 1: zona húmeda.</b> La ETR está limitada por la energía y tiende a la ETP.",
                 "<b>φ cerca de 1 o mayor: zona seca.</b> En el valle geográfico la ETR está limitada por el agua y una fracción grande de la lluvia se evapora.",
                 "<b>Norte muy lluvioso.</b> Cerca de Hidroituango, ETR/P es bajo y la mayor parte de la lluvia escurre.",
             ]))])


def s8():
    "Caudales observados"
    q = leer_csv("caudales_estaciones.csv", dtype={"codigo": str})
    if q is None:
        seccion(8, s8.__doc__, falta("caudales_estaciones.csv", 8)); return
    q = q.sort_values("area_km2")
    resumen = (f"{len(q)} estaciones del IDEAM sobre el cauce principal del Cauca, de <b>{esc(q.nombre.iloc[0])}</b> "
               f"({num(q.area_km2.iloc[0])} km²) a <b>{esc(q.nombre.iloc[-1])}</b> ({num(q.area_km2.iloc[-1])} km²).")
    spec = {"x": {"type": "log", "title": "Área de drenaje (km²)", "name": "Área", "unit": " km²", "dec": 0},
            "y": {"type": "log", "title": "Caudal medio (m³/s)"},
            "series": [{"name": "Q medio 1981–2010", "kind": "linedots", "color": "--s1", "x": vals(q.area_km2, 0),
                        "y": vals(q.Q_medio_m3s, 1), "labels": q.nombre.astype(str).tolist(), "dec": 0, "unit": " m³/s"}],
            "hover": "point", "labelEnds": True, "labelTitle": "Estación", "height": 400}
    cols = {"nombre": ("Estación", None), "codigo": ("Código", None), "area_km2": ("Área (km²)", 0),
            "inicio": ("Inicio", None), "fin": ("Fin", None), "anios_completos_1981_2010": ("Años completos 1981–2010", 0),
            "Q_medio_m3s": ("Q medio (m³/s)", 1), "q_L_s_km2": ("Rendimiento (L/s/km²)", 1), "R_obs_mm": ("Lámina (mm/año)", 0)}
    extra = (tabla(q, {k: v for k, v in cols.items() if k in q.columns}, "Estaciones aptas y caudal medio anual multianual 1981–2010")
             + grafico("g8", spec, "Caudal medio vs. área de drenaje", "Ejes logarítmicos. Pase el cursor por los puntos para ver cada estación."))
    seccion(8, s8.__doc__, resumen,
            [fig("fig_estaciones_areas_drenaje.png", "Estaciones validadas y sus áreas de drenaje anidadas (A y Q medio 1981–2010).", 8),
             fig("fig_disponibilidad_caudales.png", "Meses con caudal válido; en verde, las estaciones aptas.", 8),
             fig("fig_Q_anual_estaciones.png", "Caudal medio de cada año completo en las estaciones aptas.", 8)],
            [("Metodología", lista([
                "Caudal medio diario del IDEAM (datos abiertos, etiqueta <code>Q_MEDIA_D</code>) y coordenadas del Catálogo Nacional de Estaciones.",
                "Un mes es válido con datos en al menos el 80 % de los días; un año es completo si tiene los 12 meses.",
                "Se usan las estaciones con <b>15 o más años completos en 1981–2010</b>.",
                "Área de drenaje delimitada con el mismo DEM; caudal medio anual multianual como promedio de los años completos.",
            ])),
             ("Decisiones", lista([
                 "<b>Periodo 1981–2010.</b> Es el de la precipitación y la temperatura, y es anterior al embalse de Hidroituango.",
                 "<b>Faltantes sin rellenar.</b> Las medias cambian poco al variar el criterio de faltantes (70–95 % de días).",
                 "<b>La Pintada antes del río Arma.</b> Su área de drenaje no incluye el Arma.",
                 "<b>Estaciones excluidas.</b> Las que tienen menos de 15 años completos.",
             ]))], extra)


def s9():
    "Balance hídrico"
    if BAL is None:
        seccion(9, s9.__doc__, falta("tabla_balance_productos.csv", 9)); return
    b = BAL[(BAL.producto == "IDEAM") & (BAL.codigo != "CUENCA")].sort_values("area_km2")
    ok = b[b["cobertura_%"] >= 95]
    resumen = (f"Con la precipitación del IDEAM, el balance P − ETR reproduce el caudal observado con un "
               f"<b>error absoluto medio de {num(ok['Error_%'].abs().mean())} %</b> en {len(ok)} estaciones "
               f"(error medio {num(ok['Error_%'].mean(), 1)} %).") if len(ok) else "Balance P − ETR ≈ R por área de drenaje."
    nombres = ok.estacion.astype(str).tolist()
    spec = {"orient": "h", "x": {"type": "cat", "cats": nombres, "title": "Estación"},
            "y": {"type": "lin", "title": "Caudal medio (m³/s)"},
            "series": [{"name": "Q observado", "kind": "bar", "color": "--ink", "x": nombres, "y": vals(ok.Q_obs_m3s, 0),
                        "dec": 0, "unit": " m³/s"},
                       {"name": "Q estimado = (P − ETR)·A", "kind": "bar", "color": "--s1", "x": nombres, "y": vals(ok.Q_est_m3s, 0),
                        "dec": 0, "unit": " m³/s", "extra": [f"error {num(e, 1)} %" for e in ok["Error_%"]]}]}
    extra = (grafico("g9", spec, "Caudal observado vs. estimado", "Estaciones ordenadas de aguas arriba (arriba) a aguas abajo.")
             + tabla(b, {"estacion": ("Estación", None), "area_km2": ("Área (km²)", 0), "P_mm": ("P (mm/año)", 0),
                         "ETP_mm": ("ETP (mm/año)", 0), "ETR_mm": ("ETR (mm/año)", 0), "Q_est_mm": ("Q est. (mm/año)", 0),
                         "Q_obs_mm": ("Q obs. (mm/año)", 0), "Q_est_m3s": ("Q est. (m³/s)", 0), "Q_obs_m3s": ("Q obs. (m³/s)", 0),
                         "Error_%": ("Error (%)", 1), "cobertura_%": ("Cobertura (%)", 0)},
                     "Balance con la precipitación del IDEAM", "Cobertura: fracción del área de drenaje dentro de los mapas de la cuenca."))
    seccion(9, s9.__doc__, resumen,
            [],
            [("Metodología", lista([
                "Para periodos largos ΔS ≈ 0, así que P − ETR ≈ R y Q<sub>estimado</sub> = (P − ETR)·A.",
                "P, ETP y ETR promediadas sobre el área de drenaje de cada estación (sección 8).",
            ])),
             ("Fuentes de incertidumbre", lista([
                 "<b>Precipitación.</b> Pocas estaciones en zonas altas (efecto orográfico subestimado) y errores de captación de los pluviómetros. Q es más sensible a P que a ETP.",
                 "<b>Evapotranspiración.</b> Hargreaves solo usa temperatura y radiación extraterrestre; la curva de Budyko no tiene parámetros de vegetación ni suelo.",
                 "<b>Caudal.</b> Curvas de calibración nivel–caudal (5–15 %), vacíos, regulación de Salvajina y extracciones en el valle geográfico.",
                 "<b>Interpolación espacial.</b> Error local del orden del RMSE de validación cruzada; parte se compensa al promediar áreas grandes.",
                 "<b>Periodos.</b> Años húmedos o secos (La Niña, El Niño) que estén en un registro y no en el otro cambian el balance.",
             ]))], extra)


def s10():
    "Productos de precipitación"
    doc = leer_csv("tabla_productos_precipitacion.csv", index_col=0)
    serie = leer_csv("serie_mensual_cuenca_productos.csv", index_col=0, parse_dates=True)
    extra = ""
    if doc is not None:
        d = doc.reset_index().rename(columns={doc.index.name or "index": "Producto"})
        extra += tabla(d, {c: (c, None) for c in ["Producto", "Fuente", "Resolución espacial", "Resolución temporal",
                                                   "Periodo disponible", "Periodo seleccionado"] if c in d.columns},
                       "Documentación de los productos")
    if serie is not None:
        anual = serie.groupby(serie.index.year).sum()
        col = {"CHIRPS": "--s1", "ERA5L": "--s2", "ERA5": "--s3"}
        anios = [int(a) for a in anual.index]
        series = [{"name": NOMBRE.get(c, c), "kind": "line", "color": col.get(c, "--muted"), "x": anios,
                   "y": vals(anual[c], 0), "dec": 0, "unit": " mm"} for c in anual.columns]
        if CICLO is not None and "IDEAM" in CICLO:
            series.append({"name": "IDEAM (normal 1981–2010)", "kind": "line", "dash": True, "color": "--ink", "x": anios,
                           "y": [round(float(CICLO["IDEAM"].sum()))] * len(anios), "dec": 0, "unit": " mm"})
        extra += grafico("g10", {"x": {"type": "lin", "title": "Año", "dec": 0, "sep": False},
                                 "y": {"type": "lin", "title": "Precipitación anual de la cuenca (mm)"},
                                 "series": series, "height": 400},
                         "Precipitación anual de la cuenca por producto",
                         "La línea punteada es la normal del IDEAM; sirve de referencia, no es una serie anual.")
    seccion(10, s10.__doc__,
            "Tres productos de grilla para el <b>mismo periodo de las normales del IDEAM (1981–2010)</b>: CHIRPS v2.0 (satélite + estaciones, 0,05°), "
            "ERA5-Land (0,1°) y ERA5 (0,25°).",
            [], [("Descarga", lista([
                "<b>CHIRPS v2.0.</b> Mensual, recortado a la cuenca desde la IRI Data Library (Universidad de Columbia), en mm/mes.",
                "<b>ERA5 y ERA5-Land.</b> Promedios mensuales del Copernicus Climate Data Store. Vienen en m/día y se convierten con P = tp × 1000 × N<sub>días</sub>.",
            ])), ("Decisiones", lista([
                "<b>Periodo común 1981–2010.</b> CHIRPS comienza justamente en 1981.",
                "<b>Grilla común.</b> Todo se lleva a la grilla del DEM: bilineal para los cálculos y vecino más cercano para los mapas, que así muestran los píxeles reales.",
            ]))], extra)


def s11():
    "Mapas de precipitación por producto"
    filas = []
    if CICLO is not None:
        for c in CICLO.columns:
            filas.append({"Producto": NOMBRE.get(c, c), "P": CICLO[c].sum()})
    t = pd.DataFrame(filas)
    if len(t) and "IDEAM" in t.Producto.values:
        p0 = t.set_index("Producto").loc["IDEAM", "P"]
        t["dif"] = 100 * (t.P / p0 - 1)
    resumen = "Precipitación media anual multianual 1981–2010 de cada producto, comparada con la interpolación del IDEAM."
    if len(t) and "dif" in t:
        mayor = t[t.Producto != "IDEAM"].sort_values("dif").iloc[-1]
        resumen += f" El producto más lluvioso es <b>{esc(mayor.Producto)}</b> ({num(mayor.dif, 0)} % frente al IDEAM)."
    extra = tabla(t, {"Producto": ("Producto", None), "P": ("P media de la cuenca (mm/año)", 0), "dif": ("Diferencia con IDEAM (%)", 1)}
                  if "dif" in t else {"Producto": ("Producto", None), "P": ("P media (mm/año)", 0)})
    seccion(11, s11.__doc__, resumen,
            [fig("fig_mapas_P_productos.png", "Precipitación media anual: IDEAM y productos, con la misma escala de color.", 11),
             fig("fig_mapa_P_CHIRPS.png", "CHIRPS en su resolución nativa.", 11),
             fig("fig_mapa_P_ERA5L.png", "ERA5-Land en su resolución nativa.", 11),
             fig("fig_mapa_P_ERA5.png", "ERA5 en su resolución nativa.", 11),
             fig("fig_verificacion_productos.png", "Verificación: cada producto en toda la zona descargada, con lugares de "
                 "referencia.", 11)],
            [("Qué observar", lista([
                "Si cada producto reproduce el valle seco y el norte muy lluvioso.",
                "El suavizado por resolución: ERA5 cubre la cuenca con pocos píxeles de ≈ 28 km.",
                "La sobrestimación sobre las cordilleras, frecuente en los reanálisis sobre los Andes.",
            ]))], extra)


def s12():
    "Evaluación de los productos mediante el balance hídrico"
    if BAL is None:
        seccion(12, s12.__doc__, falta("tabla_balance_productos.csv", 12)); return
    t12 = leer_csv("tabla_punto12_cierre.csv", index_col=0)
    ev = leer_csv("evaluacion_productos_balance.csv", index_col=0)
    ci = {}
    if existe("punto12_cierre.json"):
        with open(ruta("punto12_cierre.json"), encoding="utf-8") as f:
            ci = json.load(f)
    concl = []
    if existe("conclusion_punto12.txt"):
        with open(ruta("conclusion_punto12.txt"), encoding="utf-8") as f:
            concl = [l.strip() for l in f if l.strip()]
    # Números del texto automático en formato español (coma decimal y signo menos tipográfico)
    concl = [re.sub(r"(?<![\w.])-(?=\d)", "−", re.sub(r"(\d)\.(\d)", r"\1,\2", l)) for l in concl]
    tol = ci.get("tolerancia_%", 20)

    b = BAL[(BAL.codigo != "CUENCA") & (BAL["cobertura_%"] >= 95) & BAL.Q_obs_m3s.notna()].copy()
    orden = [p for p in ["IDEAM", "CHIRPS", "ERA5-Land", "ERA5"] if p in b.producto.unique()]
    col = {"IDEAM": "--ink", "CHIRPS": "--s1", "ERA5-Land": "--s2", "ERA5": "--s3"}
    est = [str(e) for e in b.sort_values("area_km2").estacion.unique()]

    if concl:
        resumen = esc(concl[0])
    elif ev is not None:
        resumen = f"Producto con menor error en el balance: <b>{esc(ev.index[0])}</b>."
    else:
        resumen = "Balance hídrico de largo plazo repetido con la precipitación de cada producto."
    extra = ""
    if t12 is not None:
        donde = (f" · {esc(ci['estacion'])} ({num(ci['area_km2'])} km², {num(ci['fraccion_cuenca_%'])} % de la cuenca)"
                 if ci else "")
        d = t12.reset_index()
        d.columns = ["Producto"] + list(t12.columns)
        extra += tabla(d, {"Producto": ("Producto", None), "P anual (mm/año)": ("P anual (mm/año)", 0),
                           "ETR (mm/año)": ("ETR (mm/año)", 0), "Q estimado (mm/año)": ("Q estimado (mm/año)", 0),
                           "Q observado (mm/año)": ("Q observado (mm/año)", 0), "Error (%)": ("Error (%)", 1)},
                       "Tabla del enunciado en el cierre de la cuenca" + donde,
                       "Q estimado = P − ETR (Budyko con la ETP de Hargreaves). "
                       "Error = (Q estimado − Q observado)/Q observado.")
        trazas = [{"name": "Q estimado = P − ETR", "kind": "bar", "color": "--s1", "x": list(t12.index.astype(str)),
                   "y": vals(t12["Q estimado (mm/año)"], 0), "colors": [col.get(p, "--muted") for p in t12.index],
                   "dec": 0, "unit": " mm/año", "extra": [f"error {num(e, 1)} %" for e in t12["Error (%)"]]}]
        qo = float(t12["Q observado (mm/año)"].iloc[0])
        extra += grafico("g12a", {"x": {"type": "cat", "cats": list(t12.index.astype(str))},
                                  "y": {"type": "lin", "title": "Caudal (mm/año)"}, "series": trazas,
                                  "refs": [{"v": round(qo), "label": "Q obs.", "fuerte": True},
                                           {"v": round(qo * (1 + tol / 100)), "label": f"+{tol} %"},
                                           {"v": round(qo * (1 - tol / 100)), "label": f"−{tol} %"}], "height": 340},
                         "Q estimado con cada producto frente al Q observado (cierre)")
    series = []
    for p in orden:
        g = b[b.producto == p].set_index("estacion").reindex(est)
        series.append({"name": p, "kind": "bar", "color": col[p], "x": est, "y": vals(g["Error_%"], 1), "dec": 1, "unit": " %",
                       "extra": [f"Q est. {num(v)} m³/s" for v in g.Q_est_m3s]})
    extra += grafico("g12", {"orient": "h", "x": {"type": "cat", "cats": est, "title": "Estación"},
                             "y": {"type": "lin", "title": "Error en Q estimado (%)"}, "series": series,
                             "refs": [{"v": -tol, "label": f"−{tol} %"}, {"v": tol, "label": f"+{tol} %"}]},
                     "Error del balance en cada estación",
                     "Estaciones de aguas arriba (arriba) a aguas abajo. Positivo: el producto da más agua de la que mide la estación.")
    if ev is not None:
        d = ev.reset_index()
        d.columns = ["Producto"] + list(ev.columns)
        dentro = next((c for c in d.columns if c.startswith("Estaciones con |error|")), None)
        if dentro and "Estaciones evaluadas" in d:
            d[dentro] = [f"{int(a)} de {int(n)}" for a, n in zip(d[dentro], d["Estaciones evaluadas"])]
        cols = {"Producto": ("Producto", None), "Puesto": ("Puesto", 0),
                "Puntaje del balance (%)": ("Puntaje del balance (%)", 1),
                "Error de Q en el cierre (%)": ("Error en el cierre (%)", 1),
                "Error absoluto medio (%)": ("Error abs. medio en estaciones (%)", 1),
                "Error medio en estaciones (%)": ("Error medio (sesgo, %)", 1),
                "Corrección de P en el cierre (%)": ("Corrección de P necesaria (%)", 1),
                "r espacial (Q est. vs Q obs., mm)": ("r espacial", 2),
                "Estaciones con balance imposible": ("Estaciones con balance imposible", 0),
                "Dif. de P con IDEAM (%)": ("Dif. de P con IDEAM (%)", 1)}
        if dentro:
            cols[dentro] = (dentro.replace("Estaciones con ", ""), None)
        extra += tabla(d, {k: v for k, v in cols.items() if k in d.columns},
                       "Evaluación de cada producto con el balance hídrico (1 = mejor)",
                       "Puntaje = promedio entre el |error| en el cierre y el error absoluto medio en las estaciones. "
                       "La diferencia de P con el IDEAM se muestra como referencia y no entra en el puntaje.")
    bloques = []
    if concl:
        bloques.append(("Conclusión", lista([esc(l) for l in concl[1:]])))
    bloques += [
        ("Metodología", lista([
            "El balance del punto 9 se repite con la P de cada producto: la ETR de Budyko se calcula celda a celda con la P "
            "de ese producto y la misma ETP de Hargreaves.",
            "<b>Tabla del enunciado en el cierre:</b> P, ETR y Q se promedian en el área de drenaje de la estación más "
            "cercana a la presa y se comparan con su caudal observado.",
            "La misma tabla se calcula en cada estación del punto 8, de aguas arriba a aguas abajo.",
        ])),
        ("Más allá de la precipitación", lista([
            "<b>P necesaria:</b> factor que habría que aplicar a la P de cada producto para que el balance dé exactamente "
            "el caudal observado. Una corrección pequeña indica lluvia compatible con el caudal medido.",
            "<b>Balance posible:</b> la ETR «observada» = P − Q<sub>obs</sub> debe quedar entre 0 y la ETP. Fuera de ese "
            "rango, el balance no puede cerrar con esa lluvia.",
            "<b>Patrón espacial:</b> correlación entre la escorrentía estimada y la observada en las estaciones.",
            "<b>Puntaje:</b> promedio entre el error en el cierre y el error medio en las estaciones; evita que un acierto "
            "por compensación de errores en el cierre decida la evaluación.",
        ]))]
    seccion(12, s12.__doc__, resumen,
            [fig("fig_tabla12_cierre.png", "P, ETR y Q estimado con cada producto en el cierre de la cuenca.", 12),
             fig("fig_evaluacion_productos.png", "P de cada producto vs. P necesaria para cerrar el balance, y escorrentía "
                 "estimada vs. observada en las estaciones (★: cierre).", 12),
             fig("fig_balance_productos.png", "Error del balance por estación y producto, y Q estimado vs. observado.", 12),
             fig("fig_budyko_estaciones.png", "Áreas de drenaje en el espacio de Budyko con la ETR «observada» = P − Q<sub>obs</sub>.", 12),
             fig("fig_mapas_R_productos.png", "Escorrentía P − ETR con cada producto.", 12)],
            bloques, extra)


def s13():
    "Climatología de la precipitación"
    if CICLO is None or "IDEAM" not in CICLO:
        seccion(13, s13.__doc__, falta("ciclo_anual_cuenca_productos.csv", 13)); return
    ci = CICLO["IDEAM"].reindex(MESES)
    pk = picos(ci.values)
    regimen = {1: "unimodal", 2: "bimodal"}.get(len(pk), f"{len(pk)} picos")
    humedos = [m for m in MESES if ci[m] > ci.mean()]
    secos = [m for m in MESES if ci[m] <= ci.mean()]
    resumen = (f"Régimen <b>{regimen}</b>, con máximos en {', '.join(MESES[i] for i in pk)}. "
               f"Mes más lluvioso: {ci.idxmax()} ({num(ci.max())} mm); mes más seco: {ci.idxmin()} ({num(ci.min())} mm).")
    spec = {"x": {"type": "cat", "cats": MESES}, "y": {"type": "lin", "title": "Precipitación (mm/mes)"},
            "series": [{"name": "IDEAM", "kind": "bar", "color": "--s1", "x": MESES, "y": vals(ci, 0), "dec": 0, "unit": " mm",
                        "colors": ["--s1" if m in humedos else "--s1soft" for m in MESES],
                        "extra": ["mes húmedo" if m in humedos else "mes seco" for m in MESES]}],
            "refs": [{"v": round(float(ci.mean()), 1), "label": "promedio"}], "height": 380}
    extra = grafico("g13", spec, "Ciclo anual de la precipitación de la cuenca (IDEAM 1981–2010)",
                    "Barras de color intenso: meses húmedos (sobre el promedio mensual, línea punteada); atenuadas: meses secos.")
    seccion(13, s13.__doc__, resumen,
            [fig("fig_P_mensual_12mapas_IDEAM.png", "Los 12 mapas de precipitación media mensual (IDEAM).", 13),
             fig("fig_variacion_espacial_ciclo.png", "Mes de máxima lluvia, número de temporadas lluviosas e índice de estacionalidad.", 13),
             fig("fig_ciclo_por_franjas.png", "Ciclo anual en las franjas sur, centro y norte de la cuenca.", 13)],
            [("Meses", lista([f"<b>Húmedos:</b> {', '.join(humedos)}", f"<b>Secos:</b> {', '.join(secos)}"])),
             ("Interpretación", lista([
                 "El régimen bimodal se explica por el doble paso de la Zona de Convergencia Intertropical (ZCIT) sobre los Andes.",
                 "Hacia el norte (Antioquia) el periodo seco de mitad de año se debilita y el régimen tiende a unimodal.",
                 "El índice de estacionalidad y el número de temporadas lluviosas muestran dónde cambia el régimen dentro de la cuenca.",
             ]))], extra)


def s14():
    "Ciclo anual entre productos"
    met = leer_csv("metricas_ciclo_anual.csv")
    sint = leer_csv("sintesis_productos.csv", index_col=0)
    extra = ""
    if CICLO is not None:
        col = {"IDEAM": "--ink", "CHIRPS": "--s1", "ERA5-Land": "--s2", "ERA5": "--s3"}
        series = []
        for c in CICLO.columns:
            n = NOMBRE.get(c, c)
            series.append({"name": n, "kind": "linedots", "color": col.get(n, "--muted"), "x": MESES,
                           "y": vals(CICLO[c].reindex(MESES), 1), "dec": 0, "unit": " mm"})
        extra += grafico("g14", {"x": {"type": "cat", "cats": MESES}, "y": {"type": "lin", "title": "Precipitación (mm/mes)"},
                                 "series": series, "height": 400},
                         "Ciclo anual de la cuenca: IDEAM y productos")
    if met is not None:
        extra += tabla(met, {c: (h, d) for c, (h, d) in {
            "producto": ("Producto", None), "P anual (mm)": ("P anual (mm)", 0), "Bias (mm/mes)": ("Sesgo (mm/mes)", 1),
            "Bias (%)": ("Sesgo (%)", 1), "RMSE (mm/mes)": ("RMSE (mm/mes)", 1), "r": ("r", 2), "régimen": ("Régimen", None),
            "meses de máximo": ("Meses de máximo", None), "mes mínimo": ("Mes mínimo", None),
            "amplitud (máx − mín, mm)": ("Amplitud (mm)", 0)}.items() if c in met.columns}, "Métricas del ciclo anual (n = 12)")
    resumen = "Comparación de magnitud, estacionalidad, meses de máximo y mínimo y amplitud del ciclo anual, con sesgo, RMSE, r y diagrama de Taylor."
    if sint is not None and len(sint):
        resumen += f" Mejor desempeño global en esta cuenca: <b>{esc(sint.index[0])}</b>."
        s = sint.reset_index().rename(columns={sint.index.name or "index": "Producto"})
        cols = {"Producto": ("Producto", None)}
        for c in s.columns:
            if c != "Producto":
                cols[c] = (c, 2 if "puesto promedio" in c or "r ciclo" in c else (0 if "puesto" in c else 1))
        extra += tabla(s, cols, "Síntesis: ¿qué producto representa mejor cada aspecto? (1 = mejor)")
    seccion(14, s14.__doc__, resumen,
            [fig("fig_taylor_ciclo_anual.png", "Diagrama de Taylor del ciclo anual (referencia: IDEAM).", 14),
             fig("fig_diferencias_espaciales.png", "Diferencia relativa de cada producto con el mapa del IDEAM.", 14),
             fig("fig_punto_pixel.png", "Producto vs. estación (punto a píxel).", 14)],
            [("Diagrama de Taylor", lista([
                "Resume en una figura la correlación (ángulo), la desviación estándar (radio) y el RMSE centrado (distancia a la estrella).",
                "El mejor producto es el que queda más cerca de la referencia. El diagrama no muestra el sesgo, que se lee en la tabla.",
            ])),
             ("Criterios", lista([
                 "<b>Precipitación media:</b> diferencia con la P anual del IDEAM.",
                 "<b>Balance hídrico:</b> error en Q frente al caudal observado (sección 12).",
                 "<b>Ciclo anual:</b> RMSE y r del ciclo de la cuenca.",
                 "<b>Distribución espacial:</b> RMSE frente a las estaciones (punto a píxel).",
             ]))], extra)


def mapa_cuenca():
    # Genera fig_dem_cuenca.png (DEM, límite, río y punto de cierre) si no existe; requiere rasterio y geopandas
    if existe("fig_dem_cuenca.png") or not existe("DEM_cuenca.tif"):
        return
    try:
        import rasterio, matplotlib.pyplot as plt
        from matplotlib.colors import LightSource
    except ImportError as e:
        print("  Mapa de la cuenca omitido: falta", e.name); return
    try:
        with rasterio.open(ruta("DEM_cuenca.tif")) as s:
            z = s.read(1, masked=True); T = s.transform; crs = s.crs
        ext = [T.c, T.c + T.a * z.shape[1], T.f + T.e * z.shape[0], T.f]
        sombra = LightSource(315, 45).hillshade(z.filled(float(z.mean())), vert_exag=3, dx=T.a, dy=abs(T.e))
        fig_, ax = plt.subplots(figsize=(6, 9))
        ax.imshow(np.ma.masked_array(sombra, np.ma.getmaskarray(z)), extent=ext, cmap="gray")
        im = ax.imshow(z, extent=ext, cmap="terrain", alpha=0.65)
        plt.colorbar(im, ax=ax, shrink=0.6, label="Elevación (m s. n. m.)")
        for capa, dibujo in [("cuenca_cauca.gpkg", lambda g: g.boundary.plot(ax=ax, color="k", lw=0.8)),
                             ("rio_principal.gpkg", lambda g: g.plot(ax=ax, color="#08306b", lw=1))]:
            if existe(capa):
                try:
                    import geopandas as gpd
                    dibujo(gpd.read_file(ruta(capa)).to_crs(crs))
                except Exception as e:
                    print(f"  {capa} no se dibujó: {e}")
        try:
            from pyproj import Transformer
            x, y = Transformer.from_crs("EPSG:4326", crs.to_wkt(), always_xy=True).transform(CIERRE[1], CIERRE[0])
            ax.plot(x, y, "r^", ms=11, mec="k", label="Punto de cierre (Hidroituango)")
        except Exception as e:
            print("  Punto de cierre no dibujado:", e)
        ax.annotate("", xy=(0.93, 0.93), xytext=(0.93, 0.84), xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", lw=1.5))
        ax.text(0.93, 0.945, "N", transform=ax.transAxes, ha="center", fontweight="bold")
        ax.set_title("Cuenca del río Cauca hasta Hidroituango"); ax.legend(loc="lower left", fontsize=8)
        ax.text(0.02, 0.005, "MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)", transform=ax.transAxes, fontsize=7)
        ax.ticklabel_format(style="plain"); ax.tick_params(labelsize=7); ax.set_aspect("equal")
        fig_.tight_layout(); fig_.savefig(ruta("fig_dem_cuenca.png"), dpi=200); plt.close(fig_)
        print("Generado fig_dem_cuenca.png")
    except Exception as e:
        print("  No se pudo generar el mapa de la cuenca:", e)


mapa_cuenca()
for f in (s1, s2, s3, s4, s5, s6, s7, s8, s9, s10, s11, s12, s13, s14):
    construir(f)
print(f"Secciones listas: {len(SECCIONES)}")

Secciones listas: 14


## 5. Escribir la página

In [17]:
CSS = """
:root{--bg:#fcfcfb;--surface:#ffffff;--ink:#1d2433;--text:#2b3140;--muted:#6b7180;--line:#e4e5e8;--soft:#f1f3f6;
--accent:#1f4e79;--s1:#2a78d6;--s1soft:#9ec5f4;--s2:#eb6834;--s3:#1baf7a;--grid:#eceef1;color-scheme:light}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#151617;--surface:#1c1d1f;--ink:#f0efec;--text:#d9d8d3;
--muted:#9b9a93;--line:#2c2e31;--soft:#26282b;--accent:#8ab6e8;--s1:#3987e5;--s1soft:#1c5cab;--s2:#d95926;--s3:#199e70;--grid:#2a2c2f;color-scheme:dark}}
:root[data-theme="dark"]{--bg:#151617;--surface:#1c1d1f;--ink:#f0efec;--text:#d9d8d3;--muted:#9b9a93;--line:#2c2e31;--soft:#26282b;
--accent:#8ab6e8;--s1:#3987e5;--s1soft:#1c5cab;--s2:#d95926;--s3:#199e70;--grid:#2a2c2f;color-scheme:dark}
*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:16px}
body{margin:0;background:var(--bg);color:var(--text);font:16px/1.6 "IBM Plex Sans",system-ui,-apple-system,"Segoe UI",sans-serif}
h1,h2,h3{font-family:"Barlow Condensed","Arial Narrow",sans-serif;color:var(--ink);letter-spacing:.2px;margin:0}
a{color:var(--accent)}code{font-size:.88em;background:var(--soft);padding:1px 5px;border-radius:4px}
.layout{display:grid;grid-template-columns:264px minmax(0,1fr);min-height:100vh}
nav.lado{position:sticky;top:0;height:100vh;overflow:auto;border-right:1px solid var(--line);padding:28px 18px 28px 22px;background:var(--surface)}
nav.lado .marca{font:700 15px/1.25 "Barlow Condensed","Arial Narrow",sans-serif;text-transform:uppercase;letter-spacing:.8px;color:var(--muted);margin-bottom:18px}
nav.lado ol{list-style:none;margin:0;padding:0}
nav.lado li a{display:flex;gap:10px;padding:7px 10px;border-radius:8px;color:var(--text);text-decoration:none;font-size:14.5px;line-height:1.3}
nav.lado li a span{color:var(--muted);font-variant-numeric:tabular-nums;min-width:20px;text-align:right}
nav.lado li a:hover{background:var(--soft)}nav.lado li a.activo{background:var(--soft);color:var(--ink);font-weight:600}
nav.lado .cerrar{display:none}
main{padding:0 clamp(16px,4vw,56px) 80px;max-width:1320px;min-width:0}
header.cab{padding:44px 0 30px;border-bottom:1px solid var(--line);margin-bottom:8px}
header.cab h1{font-size:clamp(30px,4.2vw,46px);line-height:1.08}
header.cab p{margin:10px 0 0;color:var(--muted);font-size:15px}
.cifras{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:12px;margin-top:24px}
.cifra{background:var(--surface);border:1px solid var(--line);border-radius:12px;padding:12px 16px}
.cifra .l{font-size:13px;color:var(--muted)}.cifra .v{font-size:25px;font-weight:600;color:var(--ink);line-height:1.25}
.cifra .v small{font-size:14px;font-weight:500;color:var(--muted);margin-left:4px}
.menu-btn{display:none}
section.sec{padding:46px 0 34px;border-bottom:1px solid var(--line)}
section.sec h2{font-size:clamp(26px,3vw,34px);margin-bottom:18px}
.texto{max-width:1120px}
.texto .resumen{max-width:880px}
.bloques{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:4px 36px}
.figs{display:grid;gap:30px;margin-top:30px}
.fig-fila{display:grid;grid-template-columns:minmax(0,1.3fr) minmax(0,1fr);gap:30px;align-items:start;padding-top:28px;border-top:1px solid var(--line)}
.fig-texto .fig-num{font:600 12px/1.2 "IBM Plex Sans",system-ui,sans-serif;text-transform:uppercase;letter-spacing:1.4px;color:var(--accent);margin:6px 0 6px}
.fig-texto .fig-tit{font:600 18px/1.35 "IBM Plex Sans",system-ui,sans-serif;color:var(--ink);margin:0 0 14px;letter-spacing:0}
.fig-parte{margin-bottom:14px}
.fig-parte h4{font:600 12px/1.2 "IBM Plex Sans",system-ui,sans-serif;text-transform:uppercase;letter-spacing:1.2px;color:var(--muted);margin:0 0 5px}
.fig-parte p{margin:0;font-size:15px;line-height:1.6}
.fig-parte ul{margin:0;padding-left:18px;font-size:15px;line-height:1.55}.fig-parte li{margin:4px 0}
figure.fig{margin:0;background:var(--surface);border:1px solid var(--line);border-radius:12px;padding:10px}
figure.fig button.zoom{all:unset;cursor:zoom-in;display:block;width:100%}
figure.fig button.zoom:focus-visible{outline:2px solid var(--accent);outline-offset:3px;border-radius:6px}
figure.fig img{display:block;width:100%;height:auto;border-radius:6px;background:#fff}
.texto .resumen{font-size:18.5px;line-height:1.55;color:var(--ink);margin:0 0 22px}
.bloque{margin-bottom:20px}
.bloque h3{font:600 13px/1 "IBM Plex Sans",system-ui,sans-serif;text-transform:uppercase;letter-spacing:1.6px;color:var(--muted);margin-bottom:8px}
.bloque ul{margin:0;padding-left:18px}.bloque li{margin:6px 0}
.extra{margin-top:30px;display:grid;gap:10px;min-width:0}
h3.sub{font-size:21px;margin:18px 0 6px}
.tabla{overflow-x:auto;border:1px solid var(--line);border-radius:10px;background:var(--surface)}
table{border-collapse:collapse;width:100%;font-size:14px;font-variant-numeric:tabular-nums}
th{text-align:right;font-weight:600;color:var(--muted);padding:10px 12px;border-bottom:1px solid var(--line);white-space:nowrap;background:var(--soft)}
td{text-align:right;padding:8px 12px;border-bottom:1px solid var(--line);white-space:nowrap}
th:first-child,td.t{text-align:left}tbody tr:last-child td{border-bottom:0}tbody tr:hover td{background:var(--soft)}
.nota{font-size:13px;color:var(--muted);margin:4px 0 0}
.falta{border:1px dashed var(--line);border-radius:10px;padding:14px;color:var(--muted);font-size:14px;background:var(--soft)}
.chart{background:var(--surface);border:1px solid var(--line);border-radius:12px;padding:14px 14px 10px;min-width:0}
.leyenda{display:flex;flex-wrap:wrap;gap:6px 18px;font-size:13px;color:var(--text);margin:0 2px 10px}
.leyenda .it{display:inline-flex;align-items:center;gap:8px}
.leyenda i{display:inline-block;flex:none;background:var(--c)}
.k-bar{width:11px;height:11px;border-radius:3px}.k-dot{width:8px;height:8px;border-radius:50%}
.k-ln{width:18px;height:2px;border-radius:2px;position:relative}
.k-ln.dash{background:repeating-linear-gradient(90deg,var(--c) 0 5px,transparent 5px 8px)!important}
.k-ln.con-punto::after{content:"";position:absolute;left:5px;top:-3px;width:8px;height:8px;border-radius:50%;background:var(--c)}
.plot{position:relative}
.plot svg{display:block;touch-action:pan-y}
.plot svg text{font:12px "IBM Plex Sans",system-ui,sans-serif;fill:var(--muted)}
.plot svg text.tit{fill:var(--text);font-weight:500}.plot svg text.lab{fill:var(--text)}.plot svg text.ref{font-size:11px}
.plot svg .ov:focus-visible,.plot svg .hit:focus-visible{outline:none;stroke:var(--accent);stroke-width:1.5}
.tt{position:absolute;left:0;top:0;pointer-events:none;background:var(--surface);border:1px solid var(--line);border-radius:8px;
padding:8px 11px;font-size:13px;line-height:1.4;box-shadow:0 8px 24px rgba(0,0,0,.14);visibility:hidden;z-index:5;max-width:340px}
.tt.on{visibility:visible}.tt .h{font-weight:600;color:var(--ink);margin-bottom:3px}
.tt .r{display:flex;flex-wrap:wrap;align-items:center;column-gap:7px}.tt .r i{width:12px;height:2px;border-radius:2px;flex:none}
.tt .r b{color:var(--ink);font-weight:600;font-variant-numeric:tabular-nums}.tt .r span{color:var(--muted)}
details.datos{margin-top:8px;font-size:13px}details.datos summary{cursor:pointer;color:var(--muted);padding:2px}
details.datos .tabla{margin-top:6px;max-height:360px;overflow:auto}details.datos table{font-size:13px}
footer{padding:36px 0;color:var(--muted);font-size:14px}footer ul{padding-left:18px}
#lb{position:fixed;inset:0;background:rgba(10,12,16,.88);display:none;align-items:center;justify-content:center;z-index:50;padding:24px;cursor:zoom-out}
#lb.abierto{display:flex}#lb img{max-width:100%;max-height:100%;border-radius:8px;background:#fff}
#lb .cerrar-lb{position:absolute;top:14px;right:18px;color:#fff;font-size:15px;background:rgba(255,255,255,.12);border:0;border-radius:8px;padding:8px 12px;cursor:pointer}
@media (max-width:1000px){.layout{grid-template-columns:1fr}
nav.lado{position:fixed;inset:0 auto 0 0;width:min(320px,86vw);transform:translateX(-102%);transition:transform .2s;z-index:40;box-shadow:0 0 40px rgba(0,0,0,.25)}
nav.lado.abierto{transform:none}nav.lado .cerrar{display:inline-block;margin-bottom:14px;border:1px solid var(--line);background:var(--surface);color:var(--text);border-radius:8px;padding:6px 12px;cursor:pointer}
.menu-btn{display:inline-block;position:sticky;top:10px;z-index:30;margin-top:14px;border:1px solid var(--line);background:var(--surface);color:var(--text);border-radius:8px;padding:8px 14px;font:inherit;cursor:pointer}
.fig-fila{grid-template-columns:minmax(0,1fr);gap:16px}header.cab{padding-top:22px}}
@media (max-width:560px){.chart{padding:10px 8px 8px}.texto .resumen{font-size:17px}}
"""

# Gráficos: SVG dibujado con JavaScript propio (no necesita internet ni librerías).
JS = r"""
const NS='http://www.w3.org/2000/svg';
const ctx=document.createElement('canvas').getContext('2d');
const tw=(t,s)=>{ctx.font=(s||12)+'px "IBM Plex Sans", system-ui, sans-serif';return ctx.measureText(String(t)).width};
function nf(v,d,sep){if(v==null||!isFinite(v))return '—';
 const s=Number(v).toLocaleString('en-US',{minimumFractionDigits:d||0,maximumFractionDigits:d||0,useGrouping:sep!==false});
 return s.replace(/,/g,' ').replace('.',',').replace(/^-/,'−')}
function mk(tag,at,par){const e=document.createElementNS(NS,tag);for(const k in at)if(at[k]!=null)e.setAttribute(k,at[k]);if(par)par.appendChild(e);return e}
function he(tag,cls,t,par){const e=document.createElement(tag);if(cls)e.className=cls;if(t!=null)e.textContent=t;if(par)par.appendChild(e);return e}
const cv=c=>c&&String(c).startsWith('--')?'var('+c+')':c;
function nice(lo,hi,n){if(!(hi>lo)){const d=Math.abs(hi)||1;lo-=d*.1;hi+=d*.1}
 const raw=(hi-lo)/Math.max(n,2),p=Math.pow(10,Math.floor(Math.log10(raw))),f=raw/p,st=(f<1.5?1:f<3?2:f<7?5:10)*p;
 const a=Math.floor(lo/st+1e-9)*st,b=Math.ceil(hi/st-1e-9)*st,t=[];
 for(let k=0;a+k*st<=b+st*1e-6;k++)t.push(+(a+k*st).toPrecision(12));
 return{lo:a,hi:b,t:t,dec:Math.max(0,-Math.floor(Math.log10(st)+1e-9)),u:v=>(v-a)/(b-a)}}
function eje(tipo,v,n,cero){v=v.filter(x=>x!=null&&isFinite(x)&&(tipo!=='log'||x>0));if(!v.length)v=tipo==='log'?[1,10]:[0,1];
 if(tipo==='log'){const lo=Math.min(...v)/1.3,hi=Math.max(...v)*1.3,L=Math.log10(lo),R=Math.log10(hi);let t=[];
  for(let e=Math.floor(L);e<=Math.ceil(R);e++)for(const m of[1,2,5]){const x=m*Math.pow(10,e);if(x>=lo&&x<=hi)t.push(x)}
  const p10=t.filter(x=>Math.abs(Math.log10(x)-Math.round(Math.log10(x)))<1e-9);if(t.length>6&&p10.length>=2)t=p10;
  return{lo:lo,hi:hi,t:t,dec:0,u:x=>(Math.log10(x)-L)/(R-L)}}
 let lo=Math.min(...v),hi=Math.max(...v);if(cero){lo=Math.min(lo,0);hi=Math.max(hi,0)}return nice(lo,hi,n)}
function barra(o,c0,c1,b,e){const L=Math.abs(e-b),r=Math.min(4,L,(c1-c0)/2),s=e<b?-1:1;
 if(o==='v')return 'M'+c0+','+b+'V'+(e-s*r)+'Q'+c0+','+e+' '+(c0+r)+','+e+'H'+(c1-r)+'Q'+c1+','+e+' '+c1+','+(e-s*r)+'V'+b+'Z';
 return 'M'+b+','+c0+'H'+(e-s*r)+'Q'+e+','+c0+' '+e+','+(c0+r)+'V'+(c1-r)+'Q'+e+','+c1+' '+(e-s*r)+','+c1+'H'+b+'Z'}

function dibujar(g){
 const S=g.spec,box=g.plot,W=Math.floor(box.getBoundingClientRect().width);if(W<60)return;
 if(g.w===W&&!g.forzar)return;g.w=W;g.forzar=false;
 const viejo=box.querySelector('svg');if(viejo)viejo.remove();
 const hz=S.orient==='h',sr=S.series,cats=S.x.type==='cat'?S.x.cats:null,refs=S.refs||[];
 const yt=S.y.type||'lin',xt=S.x.type||'lin',bars=sr.filter(s=>s.kind==='bar'),nb=bars.length;
 const yv=[];sr.forEach(s=>s.y.forEach(v=>yv.push(v)));refs.forEach(r=>yv.push(r.v));
 let H,mL,mR,mT,mB,Y,X,rot=false,band=0,lab=cats?cats.slice():null;
 const fy=v=>nf(v,yt==='log'?0:Y.dec,S.y.sep);
 if(!hz){
  H=Math.round(Math.min(S.height||380,Math.max(300,W*.8)));
  Y=eje(yt,yv,Math.max(3,Math.round((H-90)/55)),nb>0||S.y.zero);
  mL=Math.ceil(Math.max(...Y.t.map(v=>tw(fy(v)))))+12;
  mR=14+(refs.some(r=>r.label)?Math.ceil(Math.max(...refs.map(r=>tw(r.label||'',11))))+8:0);
  mT=S.y.title?32:14;mB=28+(S.x.title?22:0);
  if(cats){band=(W-mL-mR)/cats.length;const mw=Math.max(...cats.map(c=>tw(c)));
   if(mw>band-8){rot=true;mB=Math.ceil(mw*.71+18)+(S.x.title?22:0);mL=Math.max(mL,Math.ceil(mw*.71-band/2+6))}}
  else{const xv=[];sr.forEach(s=>s.x.forEach(v=>xv.push(v)));X=eje(xt,xv,Math.max(3,Math.round((W-mL-mR)/90)),S.x.zero);
   const ult=X.t[X.t.length-1];if(ult!=null&&X.u(ult)>.97)mR=Math.max(mR,Math.ceil(tw(nf(ult,xt==='log'?0:X.dec,S.x.sep))/2)+4)}
 }else{
  const th=nb<=2?12:9,maxL=Math.max(80,W*.36);band=nb*th+(nb-1)*2+16;
  lab=cats.map(c=>{if(tw(c)<=maxL)return c;let t=c;while(t.length>3&&tw(t+'…')>maxL)t=t.slice(0,-1);return t+'…'});
  mL=Math.ceil(Math.max(...lab.map(c=>tw(c))))+12;mR=20;mT=refs.some(r=>r.label)?24:8;mB=28+(S.y.title?22:0);
  Y=eje(yt,yv,Math.max(3,Math.round((W-mL-mR)/85)),true);H=mT+cats.length*band+mB;
 }
 const x0=mL,x1=W-mR,y0=mT,y1=H-mB;
 if(!hz&&cats)band=(x1-x0)/cats.length;
 const svg=mk('svg',{width:W,height:H,viewBox:'0 0 '+W+' '+H,role:'img','aria-label':g.titulo||'Gráfico'});
 box.insertBefore(svg,box.firstChild);
 const pv=hz?(v=>x0+Y.u(v)*(x1-x0)):(v=>y1-Y.u(v)*(y1-y0));
 const pc=i=>hz?y0+(i+.5)*band:x0+(i+.5)*band;
 const px=v=>x0+X.u(v)*(x1-x0);
 const ci=n=>cats.indexOf(n);
 const pos=(s,j)=>cats?pc(ci(s.x[j])):px(s.x[j]);
 const Lg=mk('g',{},svg),Lh=mk('g',{},svg),Lm=mk('g',{},svg),Lf=mk('g',{},svg),Lt=mk('g',{},svg);
 // rejilla y marcas del eje de valores
 Y.t.forEach(v=>{const p=pv(v);
  if(hz){mk('line',{x1:p,x2:p,y1:y0,y2:y1,style:'stroke:var(--grid);stroke-width:1'},Lg);mk('text',{x:p,y:y1+18,'text-anchor':'middle'},Lt).textContent=fy(v)}
  else{mk('line',{x1:x0,x2:x1,y1:p,y2:p,style:'stroke:var(--grid);stroke-width:1'},Lg);mk('text',{x:x0-8,y:p+4,'text-anchor':'end'},Lt).textContent=fy(v)}});
 if(yt!=='log'&&Y.lo<0&&Y.hi>0){const p=pv(0);
  if(hz)mk('line',{x1:p,x2:p,y1:y0,y2:y1,style:'stroke:var(--muted);stroke-width:1;opacity:.7'},Lm);
  else mk('line',{x1:x0,x2:x1,y1:p,y2:p,style:'stroke:var(--muted);stroke-width:1;opacity:.7'},Lm)}
 // eje de categorías o eje x numérico
 if(cats)lab.forEach((c,i)=>{const p=pc(i);let t;
  if(hz)t=mk('text',{x:x0-8,y:p+4,'text-anchor':'end'},Lt);
  else if(rot)t=mk('text',{x:p+4,y:y1+12,'text-anchor':'end',transform:'rotate(-45 '+(p+4)+' '+(y1+12)+')'},Lt);
  else t=mk('text',{x:p,y:y1+18,'text-anchor':'middle'},Lt);
  t.textContent=c;if(c!==cats[i])mk('title',{},t).textContent=cats[i]});
 else X.t.forEach(v=>{mk('text',{x:px(v),y:y1+18,'text-anchor':'middle'},Lt).textContent=nf(v,xt==='log'?0:X.dec,S.x.sep)});
 if(S.y.title){if(hz)mk('text',{x:(x0+x1)/2,y:H-6,'text-anchor':'middle',class:'tit'},Lt).textContent=S.y.title;
  else mk('text',{x:0,y:13,'text-anchor':'start',class:'tit'},Lt).textContent=S.y.title}
 if(!hz&&S.x.title&&!cats)mk('text',{x:(x0+x1)/2,y:H-6,'text-anchor':'middle',class:'tit'},Lt).textContent=S.x.title;
 // barras agrupadas (extremo redondeado, base recta, 2 px de separación)
 if(nb){const th=hz?(nb<=2?12:9):Math.max(2,Math.min(24,(band*.72-(nb-1)*2)/nb)),gw=nb*th+(nb-1)*2,b0=pv(Math.max(Y.lo,Math.min(0,Y.hi)));
  bars.forEach((s,k)=>s.y.forEach((v,j)=>{if(v==null||!isFinite(v))return;const i=ci(s.x[j]);if(i<0)return;const c0=pc(i)-gw/2+k*(th+2);
   mk('path',{d:barra(hz?'h':'v',c0,c0+th,b0,pv(v)),style:'fill:'+cv((s.colors&&s.colors[j])||s.color)},Lm)}))}
 // líneas
 sr.forEach(s=>{if(s.kind!=='line'&&s.kind!=='linedots')return;let d='',pen=false;
  s.y.forEach((v,j)=>{if(v==null||!isFinite(v)||(yt==='log'&&v<=0)){pen=false;return}d+=(pen?'L':'M')+pos(s,j).toFixed(1)+','+pv(v).toFixed(1);pen=true});
  mk('path',{d:d,style:'fill:none;stroke:'+cv(s.color)+';stroke-width:2;stroke-linejoin:round;stroke-linecap:round'+(s.dash?';stroke-dasharray:6 4':'')},Lm)});
 // puntos (con anillo del color de fondo)
 const dots=[];
 sr.forEach(s=>{if(s.kind!=='dots'&&s.kind!=='linedots')return;s.y.forEach((v,j)=>{if(v==null||!isFinite(v)||(yt==='log'&&v<=0))return;
  const a=pos(s,j),b=pv(v);dots.push({s:s,j:j,x:a,y:b,e:mk('circle',{cx:a,cy:b,r:4,style:'fill:'+cv(s.color)+';stroke:var(--surface);stroke-width:2'},Lm)})})});
 // líneas de referencia
 refs.forEach(r=>{const p=pv(r.v),st=r.fuerte?'stroke:var(--ink);stroke-width:1.5':'stroke:var(--muted);stroke-width:1;stroke-dasharray:3 3';
  if(hz){mk('line',{x1:p,x2:p,y1:y0-2,y2:y1,style:st},Lt);if(r.label)mk('text',{x:p,y:y0-8,'text-anchor':'middle',class:'ref'},Lt).textContent=r.label}
  else{mk('line',{x1:x0,x2:x1+4,y1:p,y2:p,style:st},Lt);if(r.label)mk('text',{x:x1+8,y:p+4,'text-anchor':'start',class:'ref'},Lt).textContent=r.label}});
 // etiquetas directas en el primer y el último punto
 if(S.labelEnds){const s=sr.find(s=>s.labels);if(s){const ix=s.y.map((v,j)=>v!=null&&isFinite(v)?j:-1).filter(j=>j>=0);
  if(ix.length){const a=ix[0],b=ix[ix.length-1];
   mk('text',{x:pos(s,a)+9,y:pv(s.y[a])+16,'text-anchor':'start',class:'lab'},Lt).textContent=s.labels[a];
   if(b!==a)mk('text',{x:pos(s,b)-9,y:pv(s.y[b])-10,'text-anchor':'end',class:'lab'},Lt).textContent=s.labels[b]}}}
 // capa interactiva
 const tt=g.tt;
 function mostrar(cx,cy,h,filas){tt.replaceChildren();he('div','h',h,tt);
  filas.forEach(f=>{const r=he('div','r',null,tt),i=he('i',null,null,r);if(f.c)i.style.background=cv(f.c);
   he('b',null,f.v,r);if(f.n)he('span',null,f.n,r);if(f.e)he('span',null,'· '+f.e,r)});
  tt.style.maxWidth=Math.min(340,W-4)+'px';tt.classList.add('on');const w=tt.offsetWidth,h2=tt.offsetHeight;
  let l=cx+16;if(l+w>W)l=cx-16-w;l=Math.max(0,Math.min(l,W-w));const t=Math.max(0,Math.min(H-h2,cy-h2/2));
  tt.style.transform='translate('+Math.round(l)+'px,'+Math.round(t)+'px)'}
 function ocultar(){tt.classList.remove('on');Lh.replaceChildren();Lf.replaceChildren();dots.forEach(d=>d.e.setAttribute('r',4))}
 g.ocultar=ocultar;
 if(S.hover==='point'){
  const Lhit=mk('g',{},svg),foco=dots.length<=40;
  dots.forEach(d=>{if(d.s.hover===false)return;
   const c=mk('circle',{cx:d.x,cy:d.y,r:12,class:'hit',tabindex:foco?0:null,style:'fill:transparent'},Lhit);
   const ver=()=>{ocultar();d.e.setAttribute('r',6);
    mostrar(d.x,d.y,(d.s.labels&&d.s.labels[d.j])||d.s.name,[
     {c:null,v:nf(d.s.x[d.j],S.x.dec,S.x.sep)+(S.x.unit||''),n:S.x.name||''},
     {c:d.s.color,v:nf(d.s.y[d.j],d.s.dec)+(d.s.unit||''),n:d.s.name}])};
   c.addEventListener('pointerenter',ver);c.addEventListener('pointerdown',ver);c.addEventListener('focus',ver);
   c.addEventListener('pointerleave',e=>{if(e.pointerType!=='touch')ocultar()});c.addEventListener('blur',ocultar)});
 }else{
  const hs=sr.filter(s=>s.hover!==false);let keys;
  if(cats)keys=cats.map((c,i)=>i);else{const u=new Set();hs.forEach(s=>s.x.forEach(v=>u.add(v)));keys=[...u].sort((a,b)=>a-b)}
  const ov=mk('rect',{x:x0,y:y0,width:Math.max(1,x1-x0),height:Math.max(1,y1-y0),class:'ov',tabindex:0,style:'fill:transparent',
   'aria-label':'Use las flechas para recorrer los valores'},svg);
  let cur=-1;
  const en=k=>{cur=k;Lh.replaceChildren();Lf.replaceChildren();const key=cats?cats[k]:keys[k],p=cats?pc(k):px(key);
   if(nb){if(hz)mk('rect',{x:x0,y:p-band/2,width:x1-x0,height:band,style:'fill:var(--soft)'},Lh);
    else mk('rect',{x:p-band/2,y:y0,width:band,height:y1-y0,style:'fill:var(--soft)'},Lh)}
   else mk('line',{x1:p,x2:p,y1:y0,y2:y1,style:'stroke:var(--muted);stroke-width:1'},Lh);
   const filas=[];hs.forEach(s=>{const j=s.x.indexOf(key);if(j<0)return;const v=s.y[j],ok=v!=null&&isFinite(v);
    filas.push({c:(s.colors&&s.colors[j])||s.color,v:nf(v,s.dec)+(ok?(s.unit||''):''),n:s.name,e:s.extra?s.extra[j]:null});
    if(s.kind!=='bar'&&ok)mk('circle',{cx:p,cy:pv(v),r:5,style:'fill:'+cv(s.color)+';stroke:var(--surface);stroke-width:2'},Lf)});
   return{p:p,filas:filas,h:cats?cats[k]:nf(key,S.x.dec,S.x.sep)}};
  const mover=ev=>{const r=svg.getBoundingClientRect(),mx=ev.clientX-r.left,my=ev.clientY-r.top;let k=0;
   if(cats)k=Math.max(0,Math.min(cats.length-1,Math.floor(((hz?my:mx)-(hz?y0:x0))/band)));
   else{let best=1e9;keys.forEach((v,i)=>{const d=Math.abs(px(v)-mx);if(d<best){best=d;k=i}})}
   const o=en(k);mostrar(mx,my,o.h,o.filas)};
  const enFoco=o=>hz?mostrar(x0+(x1-x0)*.55,o.p,o.h,o.filas):mostrar(o.p,y0+(y1-y0)*.3,o.h,o.filas);
  ov.addEventListener('pointermove',mover);ov.addEventListener('pointerdown',mover);
  ov.addEventListener('pointerleave',e=>{if(e.pointerType!=='touch')ocultar()});
  ov.addEventListener('focus',()=>enFoco(en(cur<0?0:cur)));ov.addEventListener('blur',ocultar);
  ov.addEventListener('keydown',e=>{const n=keys.length;
   if(e.key==='ArrowRight'||e.key==='ArrowDown'){e.preventDefault();enFoco(en(Math.min(n-1,cur+1)))}
   else if(e.key==='ArrowLeft'||e.key==='ArrowUp'){e.preventDefault();enFoco(en(Math.max(0,cur-1)))}
   else if(e.key==='Escape')ocultar()});
 }
}

function tablaDatos(g,el){const S=g.spec,hs=S.series.filter(s=>s.hover!==false);if(!hs.length)return;
 const d=he('details','datos',null,el);he('summary',null,'Ver los datos del gráfico',d);
 const t=he('table',null,null,he('div','tabla',null,d)),trh=he('tr',null,null,he('thead',null,null,t)),tb=he('tbody',null,null,t);
 if(S.hover==='point'){
  [S.labelTitle||'Punto'].concat(hs.length>1?['Serie']:[],[(S.x.name||'x')+(S.x.unit?' ('+S.x.unit.trim()+')':''),S.y.title||'Valor'])
   .forEach(h=>he('th',null,h,trh));
  hs.forEach(s=>s.y.forEach((v,j)=>{const tr=he('tr',null,null,tb);he('td','t',s.labels?s.labels[j]:'',tr);
   if(hs.length>1)he('td','t',s.name,tr);he('td',null,nf(s.x[j],S.x.dec,S.x.sep),tr);he('td',null,nf(v,s.dec),tr)}));
 }else{
  const keys=S.x.type==='cat'?S.x.cats:[...new Set(hs.flatMap(s=>s.x))].sort((a,b)=>a-b);
  he('th',null,S.x.title||'',trh);hs.forEach(s=>he('th',null,s.name+(s.unit?' ('+s.unit.trim()+')':''),trh));
  keys.forEach(k=>{const tr=he('tr',null,null,tb);he('td','t',S.x.type==='cat'?k:nf(k,S.x.dec,S.x.sep),tr);
   hs.forEach(s=>{const j=s.x.indexOf(k);he('td',null,j<0?'—':nf(s.y[j],s.dec),tr)})});
 }}

function montar(g){const el=document.getElementById(g.id);if(!el)return;const S=g.spec;
 const ent=[],vistos=new Set();S.series.forEach(s=>{if(s.legend===false||vistos.has(s.name))return;vistos.add(s.name);ent.push(s)});
 if(ent.length>1){const L=he('div','leyenda',null,el);ent.forEach(s=>{const it=he('span','it',null,L),k=s.legendKind||s.kind;
  const i=he('i',k==='bar'?'k-bar':k==='dots'?'k-dot':'k-ln'+(k==='linedots'?' con-punto':'')+(s.dash?' dash':''),null,it);
  i.style.setProperty('--c',cv(s.color));he('span',null,s.name,it)})}
 g.plot=he('div','plot',null,el);g.tt=he('div','tt',null,g.plot);g.tt.setAttribute('aria-hidden','true');
 tablaDatos(g,el);
 const r=()=>{try{dibujar(g)}catch(e){console.error('Gráfico '+g.id,e)}};
 if(window.ResizeObserver)new ResizeObserver(()=>requestAnimationFrame(r)).observe(g.plot);else addEventListener('resize',r);
 r()}

document.addEventListener('DOMContentLoaded',()=>{
 GRAFICOS.forEach(g=>{try{montar(g)}catch(e){console.error('Gráfico '+g.id,e)}});
 document.addEventListener('pointerdown',e=>GRAFICOS.forEach(g=>{if(g.plot&&!g.plot.contains(e.target)&&g.ocultar)g.ocultar()}));
 if(document.fonts&&document.fonts.ready)document.fonts.ready.then(()=>GRAFICOS.forEach(g=>{g.forzar=true;try{dibujar(g)}catch(e){}}));
 const lb=document.getElementById('lb'),lbi=lb.querySelector('img'),nav=document.querySelector('nav.lado');
 document.querySelectorAll('button.zoom').forEach(b=>b.addEventListener('click',()=>{const i=b.querySelector('img');lbi.src=i.src;lbi.alt=i.alt;lb.classList.add('abierto')}));
 lb.addEventListener('click',()=>lb.classList.remove('abierto'));
 document.addEventListener('keydown',e=>{if(e.key==='Escape'){lb.classList.remove('abierto');nav.classList.remove('abierto')}});
 document.querySelector('.menu-btn').addEventListener('click',()=>nav.classList.add('abierto'));
 nav.querySelector('.cerrar').addEventListener('click',()=>nav.classList.remove('abierto'));
 nav.querySelectorAll('a').forEach(a=>a.addEventListener('click',()=>nav.classList.remove('abierto')));
 const links={};nav.querySelectorAll('a').forEach(a=>links[a.getAttribute('href').slice(1)]=a);
 const obs=new IntersectionObserver(es=>{es.forEach(e=>{if(e.isIntersecting){Object.values(links).forEach(a=>a.classList.remove('activo'));
  const a=links[e.target.id];if(a)a.classList.add('activo')}})},{rootMargin:'-30% 0px -60% 0px'});
 document.querySelectorAll('section.sec').forEach(s=>obs.observe(s));
 const p1=document.querySelector('section.sec');
 addEventListener('scroll',()=>{if(p1&&scrollY+innerHeight*.3<p1.offsetTop)Object.values(links).forEach(a=>a.classList.remove('activo'))},{passive:true});
});
"""


def html_seccion(s):
    k = iter(range(1, 100))
    figs = re.sub("@@N@@", lambda m: f"{s['num']}.{next(k)}", "".join(s["figuras"]))
    bloques = "".join(f'<div class="bloque"><h3>{esc(t)}</h3>{c}</div>' for t, c in s["bloques"] if c)
    texto = (f'<div class="texto"><p class="resumen">{s["resumen"]}</p>'
             + (f'<div class="bloques">{bloques}</div>' if bloques else "") + "</div>")
    cuerpo = texto + (f'<div class="figs">{figs}</div>' if figs else "")
    extra = f'<div class="extra">{s["extra"]}</div>' if s["extra"] else ""
    return f'<section class="sec" id="{s["id"]}"><h2>{s["num"]}. {esc(s["titulo"])}</h2>{cuerpo}{extra}</section>'


# Cifras principales de la cuenca (con la precipitación del IDEAM)
cifras = ""
if CUENCA is not None and "IDEAM" in CUENCA.index:
    c = CUENCA.loc["IDEAM"]
    datos_c = [("Área de la cuenca", num(c.get("area_km2")), "km²"), ("Precipitación (IDEAM)", num(c.get("P_mm")), "mm/año"),
               ("ETP (Hargreaves)", num(c.get("ETP_mm")), "mm/año"), ("ETR (Budyko)", num(c.get("ETR_mm")), "mm/año"),
               ("Escorrentía P − ETR", num(c.get("Q_est_mm")), "mm/año")]
    cifras = '<div class="cifras">' + "".join(
        f'<div class="cifra"><div class="l">{esc(l)}</div><div class="v">{v}<small>{esc(u)}</small></div></div>'
        for l, v, u in datos_c if v != "—") + "</div>"

SECCIONES.sort(key=lambda s: s["num"])
nav = "".join(f'<li><a href="#{s["id"]}"><span>{s["num"]}</span>{esc(s["titulo"])}</a></li>' for s in SECCIONES)
fuentes = lista([
    "IDEAM – Promedios climatológicos 1981–2010 (precipitación y temperatura).",
    "IDEAM – Caudales medios diarios de los ríos de Colombia y Catálogo Nacional de Estaciones (datos.gov.co).",
    "Funk et al. (2015) – CHIRPS v2.0, Climate Hazards Center (UCSB); descarga: IRI Data Library.",
    "Muñoz-Sabater et al. (2021) – ERA5-Land; Hersbach et al. (2020) – ERA5. Copernicus Climate Data Store.",
    "Hargreaves y Samani (1985); Budyko (1974); Taylor (2001).",
])
datos_js = json.dumps(GRAFICOS, ensure_ascii=False).replace("</", "<\\/")
pagina = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Balance hídrico Cauca–Hidroituango</title>
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@600;700&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<style>{CSS}</style></head><body>
<div class="layout">
<nav class="lado" aria-label="Secciones"><button class="cerrar" type="button">Cerrar</button>
<div class="marca">Trabajo 1 · Hidroclimatología</div><ol>{nav}</ol></nav>
<main><button class="menu-btn" type="button">☰ Secciones</button>
<header class="cab"><h1>{esc(TITULO)}</h1><p>{esc(SUBTITULO)} · {esc(AUTORES)}<br>
Mapas: clic para ampliar · gráficos: pase el cursor (o toque) para ver los valores</p>{cifras}</header>
{"".join(html_seccion(s) for s in SECCIONES)}
<footer><h3>Fuentes de información</h3>{fuentes}<p>Página generada el {datetime.date.today():%d/%m/%Y}.</p></footer>
</main></div>
<div id="lb" role="dialog" aria-label="Imagen ampliada"><button class="cerrar-lb" type="button">Cerrar ✕</button><img alt=""></div>
<script>const GRAFICOS={datos_js};
{JS}</script>
</body></html>"""
with open(os.path.join(SALIDA, "index.html"), "w", encoding="utf-8") as f:
    f.write(pagina)
tam = sum(os.path.getsize(os.path.join(dp, fn)) for dp, _, fns in os.walk(SALIDA) for fn in fns) / 1e6
print(f"Listo: {os.path.abspath(os.path.join(SALIDA, 'index.html'))}  ({tam:.1f} MB en total, {len(GRAFICOS)} gráficos)")

Listo: c:\Users\User\Desktop\Unal\Hidrometeorologia\Trabajo 1\Desarrollo\04_PY\sitio\index.html  (9.9 MB en total, 8 gráficos)


In [18]:
# Abrir la página en el navegador
import webbrowser
webbrowser.open("file://" + os.path.abspath(os.path.join(SALIDA, "index.html")))

True

## 6. Publicarla en internet (GitHub Pages)
Así quedó publicada la página de su compañera (`usuario.github.io/nombre-del-repositorio`).
1. Cree una cuenta en **github.com**, si no tiene.
2. Cree un repositorio nuevo **público**, por ejemplo `balance-hidrico-cauca`.
3. En el repositorio, elija *Add file → Upload files* y arrastre **el contenido** de la carpeta `sitio`: el archivo
   `index.html` y la carpeta `img`. Luego dé *Commit changes*.
4. Vaya a *Settings → Pages*. En *Branch* elija `main` y la carpeta `/ (root)`, y dé *Save*.
5. En uno o dos minutos la página queda en `https://SU-USUARIO.github.io/balance-hidrico-cauca/`.

Si cambia algún resultado, vuelva a correr este notebook y suba de nuevo los archivos de `sitio`.